# Universal Krea 2 LoRA Training, Evaluation, and Export Pipeline — Reliability Revision 2.1

This Google Colab notebook provides a reusable end-to-end pipeline for training a Krea 2 LoRA with AI Toolkit. It is designed for character, style, object, product, and other captioned concept datasets rather than one identity or one training history.

The default workflow trains on `krea/Krea-2-Raw` and evaluates on `krea/Krea-2-Turbo`. All project-specific settings are centralized. The notebook discovers the actual dataset size and actual checkpoints produced by the current run, supports smoke-only, interrupted, resumed, and completed training states, and never chooses a best checkpoint automatically.

Reliability Revision 2.1 carries forward the failures found during the real Colab implementation: exact-revision fetching, Colab-safe virtual-environment creation, incomplete-training checkpoint discovery, BF16-aware live adapter scaling, non-destructive LoRA loading, and conservative completion claims.

Run the notebook in order. Edit the central configuration cell before starting. Runtime files remain under `/content`; Google Drive is not used.


## What was generalized and what was repaired

This rewrite removes project-specific assumptions and carries forward the fixes proven during interactive development:

- No fixed trigger word, identity name, dataset size, run name, or project filename.
- No fixed training length, checkpoint interval, or required checkpoint list.
- No assumption that production training has completed.
- Smoke-only runs and intentionally interrupted runs can proceed to structural inference testing.
- Checkpoint sweeps are selected dynamically from the files that actually exist.
- No predetermined refinement window or provisional winner.
- Prompts, seeds, resolution, sampling settings, adapter names, adapter scales, and optional additional LoRAs are editable.
- Nested dataset folders and repeated basenames in different folders are canonicalized safely.
- Exact Git commits are fetched before checkout, including shallow-clone fallback handling.
- Colab environment creation uses `virtualenv` instead of relying on the runtime's failing `ensurepip` path.
- Broken or partially created environments are detected and rebuilt automatically.
- Interrupted runs do not require a complete checkpoint sequence and are not labeled production-complete.
- Live LoRA scaling is validated through AI Toolkit's `torch_multiplier` path with the actual BF16 representation.
- Packaging discovers the current run’s actual checkpoints, training state, manifests, logs, dataset, and generated comparisons.
- Every run records the resolved AI Toolkit commit, Diffusers commit, model revisions, hashes, configuration, and package freeze.

A partial or smoke-only run is never labeled production-complete, and visual quality selection always remains a manual decision.


## Verified implementation baseline

This notebook was rewritten against the official Krea 2 repository, official Krea model cards, and the current official AI Toolkit Krea 2 integration available on 21 July 2026.

The installation cell resolves and records the exact AI Toolkit commit used at runtime. Set `ai_toolkit_revision` to an exact commit for strict reproducibility, or leave it as `main` to resolve the current source and preserve that resolved commit in the run manifest.

The notebook verifies the active Krea 2 implementation before downloading model assets or training. It expects the Krea 2 architecture identifier, `SingleStreamDiT` LoRA target, Qwen3-VL text encoder, Qwen-Image VAE, and AI Toolkit LoRA conversion hooks to be present.

## Supported execution modes

The same notebook supports four workflows without editing downstream cells:

1. **Training smoke test only:** enable `run_training_smoke_test` and disable `run_production_training`.
2. **Interrupted production test:** start production training, stop it after a checkpoint finishes saving, then continue with checkpoint inventory and inference.
3. **Completed production run:** allow training to reach its configured step count and evaluate any discovered checkpoints.
4. **Resume:** rerun the production-training cell with the same run name, training directory, configuration, and optimizer state.

Inference from a smoke or interrupted checkpoint proves pipeline compatibility only. It does not authorize a final quality claim.

## Cell 1 — Central configuration

**Code cell.** Edit this cell before running the rest of the notebook. Every project-specific value is centralized here.

In [ ]:
import json
import re
from pathlib import Path

PROJECT_ROOT = Path("/content/krea2_lora").resolve()

USER_CONFIG = {
    "project_name": "my_krea2_lora_project",
    "run_name": "my_krea2_lora_v1",
    "concept_type": "character",
    "trigger_word": "myconcept",
    "expected_pair_count": None,
    "minimum_pair_count": 4,
    "caption_trigger_policy": "require",
    "auto_prefix_missing_trigger": False,
    "fail_on_exact_duplicates": True,
    "near_duplicate_hamming_threshold": 8,
    "ai_toolkit_repository": "https://github.com/ostris/ai-toolkit.git",
    "ai_toolkit_revision": "main",
    "force_reinstall_environment": False,
    "virtualenv_version": "21.6.1",
    "torch_version": "2.9.1",
    "torchvision_version": "0.24.1",
    "torchaudio_version": "2.9.1",
    "torch_index_url": "https://download.pytorch.org/whl/cu128",
    "training_model_repository": "krea/Krea-2-Raw",
    "training_model_revision": None,
    "training_checkpoint_filename": "raw.safetensors",
    "inference_model_repository": "krea/Krea-2-Turbo",
    "inference_model_revision": None,
    "inference_checkpoint_filename": "turbo.safetensors",
    "text_encoder_repository": "Qwen/Qwen3-VL-4B-Instruct",
    "text_encoder_revision": None,
    "vae_repository": "Qwen/Qwen-Image",
    "vae_revision": None,
    "max_text_length": 512,
    "training_resolutions": [768, 1024],
    "batch_size": 1,
    "gradient_accumulation": 1,
    "training_steps": 2000,
    "learning_rate": 0.0001,
    "weight_decay": 0.0001,
    "optimizer": "adamw",
    "lr_scheduler": "constant",
    "max_grad_norm": 1.0,
    "lora_rank": 32,
    "lora_alpha": 32,
    "save_every": 200,
    "max_step_saves_to_keep": 50,
    "dataset_repeats": 1,
    "caption_dropout_rate": 0.0,
    "token_dropout_rate": 0.0,
    "shuffle_tokens": False,
    "keep_tokens": 1,
    "flip_x": False,
    "cache_latents_to_disk": True,
    "cache_text_embeddings": True,
    "training_dtype": "bf16",
    "quantize_transformer": False,
    "quantize_text_encoder": False,
    "low_vram": False,
    "layer_offloading": False,
    "run_training_smoke_test": True,
    "smoke_test_steps": 3,
    "run_production_training": True,
    "disable_training_samples": True,
    "training_sample_every": 200,
    "raw_sample_steps": 52,
    "raw_sample_guidance": 3.5,
    "evaluation_prompts": [
        "myconcept in a neutral studio photograph with clear subject visibility",
        "myconcept in a natural outdoor environment with realistic lighting",
        "myconcept in a wide composition with a different pose or presentation",
    ],
    "evaluation_seeds": [42, 12345, 987654321],
    "inference_width": 1024,
    "inference_height": 1024,
    "inference_steps": 8,
    "inference_guidance": 0.0,
    "negative_prompt": "",
    "primary_adapter_name": "concept_adapter",
    "primary_adapter_scale": 1.0,
    "additional_loras": [],
    "checkpoint_selection_mode": "final_if_present_else_latest",
    "manual_checkpoint_step": None,
    "checkpoint_sweep_mode": "auto",
    "manual_sweep_steps": [],
    "maximum_sweep_checkpoints": 8,
    "scale_sweep": [0.6, 0.8, 1.0],
    "run_inference": True,
    "run_checkpoint_sweep": True,
    "run_scale_sweep": True,
    "checkpoints_per_archive": 4,
    "auto_download_exports": False,
    "strict_hardware_check": False,
    "minimum_gpu_memory_gib": 40,
}

if not re.fullmatch(r"[A-Za-z0-9._-]+", USER_CONFIG["run_name"]):
    raise RuntimeError("run_name may contain only letters, numbers, periods, underscores, and hyphens.")

if USER_CONFIG["caption_trigger_policy"] not in {"require", "warn", "ignore"}:
    raise RuntimeError("caption_trigger_policy must be require, warn, or ignore.")

if USER_CONFIG["checkpoint_selection_mode"] not in {"final_if_present_else_latest", "latest", "manual"}:
    raise RuntimeError("checkpoint_selection_mode is invalid.")

if USER_CONFIG["checkpoint_sweep_mode"] not in {"auto", "all", "manual", "selected_only"}:
    raise RuntimeError("checkpoint_sweep_mode is invalid.")

positive_integer_fields = [
    "minimum_pair_count",
    "max_text_length",
    "batch_size",
    "gradient_accumulation",
    "training_steps",
    "lora_rank",
    "lora_alpha",
    "save_every",
    "max_step_saves_to_keep",
    "dataset_repeats",
    "smoke_test_steps",
    "inference_width",
    "inference_height",
    "inference_steps",
    "maximum_sweep_checkpoints",
    "checkpoints_per_archive",
]
for field in positive_integer_fields:
    if not isinstance(USER_CONFIG[field], int) or USER_CONFIG[field] <= 0:
        raise RuntimeError(f"{field} must be a positive integer.")

if USER_CONFIG["expected_pair_count"] is not None:
    if not isinstance(USER_CONFIG["expected_pair_count"], int) or USER_CONFIG["expected_pair_count"] <= 0:
        raise RuntimeError("expected_pair_count must be null or a positive integer.")

if not USER_CONFIG["trigger_word"].strip() and USER_CONFIG["caption_trigger_policy"] == "require":
    raise RuntimeError("A non-empty trigger_word is required when caption_trigger_policy is require.")

if not USER_CONFIG["evaluation_prompts"] or not all(isinstance(prompt, str) and prompt.strip() for prompt in USER_CONFIG["evaluation_prompts"]):
    raise RuntimeError("evaluation_prompts must contain at least one non-empty prompt.")

if not USER_CONFIG["evaluation_seeds"] or not all(isinstance(seed, int) for seed in USER_CONFIG["evaluation_seeds"]):
    raise RuntimeError("evaluation_seeds must contain at least one integer seed.")

if not USER_CONFIG["training_resolutions"] or not all(isinstance(value, int) and value > 0 for value in USER_CONFIG["training_resolutions"]):
    raise RuntimeError("training_resolutions must contain positive integers.")

if not USER_CONFIG["scale_sweep"] or not all(isinstance(value, (int, float)) and value >= 0 for value in USER_CONFIG["scale_sweep"]):
    raise RuntimeError("scale_sweep must contain one or more non-negative values.")

if not isinstance(USER_CONFIG["primary_adapter_scale"], (int, float)) or USER_CONFIG["primary_adapter_scale"] < 0:
    raise RuntimeError("primary_adapter_scale must be non-negative.")

if USER_CONFIG["trigger_word"]:
    USER_CONFIG["evaluation_prompts"] = [
        prompt.replace("myconcept", USER_CONFIG["trigger_word"])
        for prompt in USER_CONFIG["evaluation_prompts"]
    ]

adapter_names = {USER_CONFIG["primary_adapter_name"]}
if not USER_CONFIG["primary_adapter_name"].strip():
    raise RuntimeError("primary_adapter_name must not be empty.")
for item in USER_CONFIG["additional_loras"]:
    if not isinstance(item, dict):
        raise RuntimeError("Every additional_loras entry must be a dictionary.")
    missing = {"name", "path", "scale"} - set(item)
    if missing:
        raise RuntimeError(f"An additional LoRA entry is missing fields: {sorted(missing)}")
    if item["name"] in adapter_names:
        raise RuntimeError(f"Duplicate adapter name: {item['name']}")
    if float(item["scale"]) < 0:
        raise RuntimeError(f"Additional LoRA scale must be non-negative: {item['name']}")
    adapter_names.add(item["name"])

PATHS = {
    "root": PROJECT_ROOT,
    "ai_toolkit": PROJECT_ROOT / "ai-toolkit",
    "venv": PROJECT_ROOT / "venv",
    "venv_python": PROJECT_ROOT / "venv" / "bin" / "python",
    "config": PROJECT_ROOT / "config",
    "logs": PROJECT_ROOT / "logs",
    "models": PROJECT_ROOT / "models",
    "dataset": PROJECT_ROOT / "dataset",
    "dataset_raw": PROJECT_ROOT / "dataset" / "raw",
    "dataset_training": PROJECT_ROOT / "dataset" / "training",
    "dataset_audit": PROJECT_ROOT / "dataset" / "audit",
    "checkpoints": PROJECT_ROOT / "checkpoints",
    "smoke_checkpoints": PROJECT_ROOT / "smoke_checkpoints",
    "inference": PROJECT_ROOT / "inference",
    "exports": PROJECT_ROOT / "exports",
    "helpers": PROJECT_ROOT / "runtime_helpers",
}

for key in [
    "root",
    "config",
    "logs",
    "models",
    "dataset",
    "dataset_audit",
    "checkpoints",
    "smoke_checkpoints",
    "inference",
    "exports",
    "helpers",
]:
    PATHS[key].mkdir(parents=True, exist_ok=True)

configuration_path = PATHS["config"] / "user_configuration.json"
configuration_path.write_text(json.dumps(USER_CONFIG, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(USER_CONFIG, indent=2, ensure_ascii=False))
print(f"Configuration saved to: {configuration_path}")

## Cell 2 — Verify the runtime

**Code cell.** Verifies CUDA, BF16 support, GPU memory, Python, disk space, and the temporary-storage policy. A non-A100 GPU is allowed unless strict hardware checking is enabled.

In [ ]:
import json
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime before continuing.")

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The selected GPU does not support BF16.")

gpu_properties = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu_properties.total_memory / (1024 ** 3)
disk = shutil.disk_usage(PROJECT_ROOT)

runtime = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_memory_gib": gpu_memory_gib,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "cuda_runtime": torch.version.cuda,
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "disk_total_gib": disk.total / (1024 ** 3),
    "disk_free_gib": disk.free / (1024 ** 3),
    "project_root": str(PROJECT_ROOT),
    "google_drive_used": False,
}

if USER_CONFIG["strict_hardware_check"] and gpu_memory_gib < USER_CONFIG["minimum_gpu_memory_gib"]:
    raise RuntimeError(
        f"GPU memory is {gpu_memory_gib:.2f} GiB, below the configured minimum of "
        f"{USER_CONFIG['minimum_gpu_memory_gib']} GiB."
    )

runtime_path = PATHS["config"] / "runtime_manifest.json"
runtime_path.write_text(json.dumps(runtime, indent=2), encoding="utf-8")
print(json.dumps(runtime, indent=2))
print(f"Runtime manifest: {runtime_path}")

## Cell 3 — Install AI Toolkit in an isolated environment

**Code cell.** Clones or updates the official repository, fetches the requested branch or exact commit before checkout, records the resolved commit, and creates a Colab-safe isolated environment with the configured `virtualenv` release.

The cell does not use `python -m venv`, because some Colab Python images fail inside `ensurepip`. It detects and removes a partially created or unusable environment, rebuilds it with `virtualenv`, installs the official CUDA PyTorch stack and repository requirements, verifies the pinned Diffusers commit from the active requirements, and records a complete package freeze.


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

repository_path = PATHS["ai_toolkit"]
venv_path = PATHS["venv"]
venv_python = PATHS["venv_python"]
log_path = PATHS["logs"] / "environment_installation.log"
manifest_path = PATHS["config"] / "source_environment_manifest.json"
freeze_path = PATHS["config"] / "installed_packages.txt"

log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text("", encoding="utf-8")


def run_command(command, cwd=None, allow_failure=False):
    command = [str(part) for part in command]
    command_text = " ".join(command)
    print(f"$ {command_text}")
    captured = []
    with log_path.open("a", encoding="utf-8") as log_handle:
        log_handle.write(f"\n$ {command_text}\n")
        log_handle.flush()
        process = subprocess.Popen(
            command,
            cwd=str(cwd) if cwd is not None else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        if process.stdout is None:
            raise RuntimeError(f"Unable to capture command output: {command_text}")
        for line in process.stdout:
            print(line, end="")
            captured.append(line)
            log_handle.write(line)
            log_handle.flush()
        return_code = process.wait()
        log_handle.write(f"\nExit code: {return_code}\n")
        log_handle.flush()
    result = {
        "command": command,
        "returncode": return_code,
        "stdout": "".join(captured),
    }
    if return_code != 0 and not allow_failure:
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {command_text}\n"
            f"Complete log: {log_path}"
        )
    return result


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def environment_is_usable():
    if not venv_python.is_file():
        return False
    result = subprocess.run(
        [str(venv_python), "-c", "import pip, sys; print(sys.executable)"],
        capture_output=True,
        text=True,
    )
    return result.returncode == 0


if repository_path.exists() and not (repository_path / ".git").is_dir():
    shutil.rmtree(repository_path)

if not repository_path.exists():
    run_command(["git", "init", str(repository_path)])

remote = run_command(
    ["git", "remote", "get-url", "origin"],
    cwd=repository_path,
    allow_failure=True,
)

if remote["returncode"] == 0:
    current_remote = remote["stdout"].strip()
    if current_remote != USER_CONFIG["ai_toolkit_repository"]:
        run_command(
            ["git", "remote", "set-url", "origin", USER_CONFIG["ai_toolkit_repository"]],
            cwd=repository_path,
        )
else:
    run_command(
        ["git", "remote", "add", "origin", USER_CONFIG["ai_toolkit_repository"]],
        cwd=repository_path,
    )

revision = USER_CONFIG["ai_toolkit_revision"]
if revision in {"main", "master"}:
    run_command(
        ["git", "fetch", "--no-tags", "--depth=1", "origin", revision],
        cwd=repository_path,
    )
    run_command(
        ["git", "checkout", "--detach", "--force", "FETCH_HEAD"],
        cwd=repository_path,
    )
else:
    direct_fetch = run_command(
        ["git", "fetch", "--no-tags", "--depth=1", "origin", revision],
        cwd=repository_path,
        allow_failure=True,
    )
    if direct_fetch["returncode"] != 0:
        run_command(
            ["git", "fetch", "--no-tags", "origin", "+refs/heads/*:refs/remotes/origin/*"],
            cwd=repository_path,
        )
    object_check = run_command(
        ["git", "cat-file", "-e", f"{revision}^{{commit}}"],
        cwd=repository_path,
        allow_failure=True,
    )
    if object_check["returncode"] != 0:
        raise RuntimeError(f"The configured AI Toolkit revision could not be fetched: {revision}")
    run_command(
        ["git", "checkout", "--detach", "--force", revision],
        cwd=repository_path,
    )

run_command(["git", "reset", "--hard", "HEAD"], cwd=repository_path)
run_command(["git", "clean", "-ffdx"], cwd=repository_path)
run_command(["git", "submodule", "sync", "--recursive"], cwd=repository_path)
run_command(
    ["git", "submodule", "update", "--init", "--recursive", "--depth=1"],
    cwd=repository_path,
)
resolved_commit = run_command(
    ["git", "rev-parse", "HEAD"],
    cwd=repository_path,
)["stdout"].strip()

required_entries = [
    repository_path / "run.py",
    repository_path / "requirements.txt",
    repository_path / "requirements_base.txt",
    repository_path / "toolkit",
    repository_path / "extensions_built_in" / "diffusion_models" / "krea2" / "krea2.py",
]
for required_entry in required_entries:
    if not required_entry.exists():
        raise RuntimeError(f"Required AI Toolkit entry is missing: {required_entry}")

requirements_path = repository_path / "requirements.txt"
requirements_base_path = repository_path / "requirements_base.txt"
requirements_hash = sha256_file(requirements_path)
requirements_base_hash = sha256_file(requirements_base_path)
requirements_base_text = requirements_base_path.read_text(encoding="utf-8")
diffusers_match = re.search(
    r"diffusers\.git@([0-9a-fA-F]{40})",
    requirements_base_text,
)
expected_diffusers_commit = diffusers_match.group(1).lower() if diffusers_match else None

if USER_CONFIG["force_reinstall_environment"] and venv_path.exists():
    shutil.rmtree(venv_path)

if venv_path.exists() and not environment_is_usable():
    shutil.rmtree(venv_path)

if not environment_is_usable():
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            f"virtualenv=={USER_CONFIG['virtualenv_version']}",
        ]
    )
    run_command(
        [
            sys.executable,
            "-m",
            "virtualenv",
            "--python",
            sys.executable,
            str(venv_path),
        ]
    )

if not environment_is_usable():
    raise RuntimeError(f"The isolated environment is unusable after creation: {venv_path}")

run_command(
    [
        str(venv_python),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    cwd=repository_path,
)
run_command(
    [
        str(venv_python),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        f"torch=={USER_CONFIG['torch_version']}",
        f"torchvision=={USER_CONFIG['torchvision_version']}",
        f"torchaudio=={USER_CONFIG['torchaudio_version']}",
        "--index-url",
        USER_CONFIG["torch_index_url"],
    ],
    cwd=repository_path,
)
run_command(
    [
        str(venv_python),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "-r",
        str(requirements_path),
    ],
    cwd=repository_path,
)
run_command([str(venv_python), "-m", "pip", "check"], cwd=repository_path)

verification_script = r"""
import importlib.metadata
import json
import sys
import torch
import torchvision
import torchaudio
import diffusers

distribution = importlib.metadata.distribution("diffusers")
direct_url_text = distribution.read_text("direct_url.json")
direct_url = json.loads(direct_url_text) if direct_url_text else {}
result = {
    "python": sys.version.split()[0],
    "python_executable": sys.executable,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torchaudio": torchaudio.__version__,
    "cuda_runtime": torch.version.cuda,
    "diffusers": diffusers.__version__,
    "diffusers_commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "cuda_available": torch.cuda.is_available(),
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(result))
"""
verification_result = run_command(
    [str(venv_python), "-c", verification_script],
    cwd=repository_path,
)
verification_lines = [line for line in verification_result["stdout"].splitlines() if line.strip()]
if not verification_lines:
    raise RuntimeError("The environment verification returned no structured output.")
verification = json.loads(verification_lines[-1])

if expected_diffusers_commit is not None:
    installed_diffusers_commit = verification.get("diffusers_commit")
    if installed_diffusers_commit != expected_diffusers_commit:
        raise RuntimeError(
            "The installed Diffusers commit does not match the active AI Toolkit requirements.\n"
            f"Expected: {expected_diffusers_commit}\n"
            f"Received: {installed_diffusers_commit}"
        )

freeze = run_command(
    [str(venv_python), "-m", "pip", "freeze"],
    cwd=repository_path,
)["stdout"]
freeze_path.write_text(freeze, encoding="utf-8")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": USER_CONFIG["ai_toolkit_repository"],
    "requested_revision": revision,
    "resolved_commit": resolved_commit,
    "repository_path": str(repository_path),
    "requirements_path": str(requirements_path),
    "requirements_sha256": requirements_hash,
    "requirements_base_path": str(requirements_base_path),
    "requirements_base_sha256": requirements_base_hash,
    "expected_diffusers_commit": expected_diffusers_commit,
    "virtualenv_version": USER_CONFIG["virtualenv_version"],
    "environment_creation_method": "virtualenv",
    "system_python": sys.executable,
    "venv_python": str(venv_python),
    "environment_verification": verification,
    "package_freeze": str(freeze_path),
    "installation_log": str(log_path),
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(json.dumps(manifest, indent=2))
print("AI Toolkit environment installation passed.")


## Cell 4 — Verify the active Krea 2 implementation

**Code cell.** Imports the actual installed classes and verifies architecture identifiers, LoRA targets, text encoder defaults, VAE defaults, LoRA conversion hooks, CUDA access, package versions, and the exact Diffusers commit declared by the checked-out AI Toolkit requirements. It fails instead of silently continuing when the installed source no longer matches the expected Krea 2 contract.


In [ ]:
import importlib.metadata
import json
import os
import subprocess

source_environment_manifest = json.loads(
    (PATHS["config"] / "source_environment_manifest.json").read_text(encoding="utf-8")
)
expected_diffusers_commit = source_environment_manifest.get("expected_diffusers_commit")

verification_script = r"""
import importlib.metadata
import inspect
import json
import torch
import diffusers
import transformers
import accelerate
from diffusers import AutoencoderKLQwenImage
from toolkit.config_modules import ModelConfig
from extensions_built_in.diffusion_models.krea2.krea2 import Krea2Model, QWEN3_VL_PATH, QWEN_IMAGE_VAE_PATH

distribution = importlib.metadata.distribution("diffusers")
direct_url_text = distribution.read_text("direct_url.json")
direct_url = json.loads(direct_url_text) if direct_url_text else {}
probe_configuration = ModelConfig(
    name_or_path="unused",
    arch="krea2",
    dtype="bf16",
    vae_dtype="bf16",
    te_dtype="bf16",
    model_kwargs={},
)
probe = Krea2Model(device="cpu", model_config=probe_configuration, dtype="bf16")
result = {
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "diffusers": diffusers.__version__,
    "diffusers_commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "model_class": f"{Krea2Model.__module__}.{Krea2Model.__name__}",
    "arch": probe.arch,
    "target_lora_modules": list(probe.target_lora_modules),
    "vae_scale_factor": probe.vae_scale_factor,
    "text_encoder_default": QWEN3_VL_PATH,
    "vae_default": QWEN_IMAGE_VAE_PATH,
    "vae_class": f"{AutoencoderKLQwenImage.__module__}.{AutoencoderKLQwenImage.__name__}",
    "has_save_conversion": hasattr(Krea2Model, "convert_lora_weights_before_save"),
    "has_load_conversion": hasattr(Krea2Model, "convert_lora_weights_before_load"),
    "source_file": inspect.getsourcefile(Krea2Model),
}
print(json.dumps(result))
"""

environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
result = subprocess.run(
    [str(PATHS["venv_python"]), "-c", verification_script],
    cwd=str(PATHS["ai_toolkit"]),
    env=environment,
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(f"Krea 2 implementation verification failed.\n{result.stdout}\n{result.stderr}")

verification = json.loads(result.stdout.strip().splitlines()[-1])
expected_torch_prefix = USER_CONFIG["torch_version"]
if not verification["torch"].startswith(expected_torch_prefix):
    raise RuntimeError(
        f"PyTorch version mismatch. Expected {expected_torch_prefix}, received {verification['torch']}."
    )
if expected_diffusers_commit is not None and verification["diffusers_commit"] != expected_diffusers_commit:
    raise RuntimeError(
        "Diffusers commit mismatch.\n"
        f"Expected: {expected_diffusers_commit}\n"
        f"Received: {verification['diffusers_commit']}"
    )
if verification["arch"] != "krea2":
    raise RuntimeError(f"Unexpected Krea 2 architecture identifier: {verification['arch']}")
if verification["target_lora_modules"] != ["SingleStreamDiT"]:
    raise RuntimeError(f"Unexpected Krea 2 LoRA targets: {verification['target_lora_modules']}")
if verification["text_encoder_default"] != "Qwen/Qwen3-VL-4B-Instruct":
    raise RuntimeError(f"Unexpected Krea 2 text encoder default: {verification['text_encoder_default']}")
if verification["vae_default"] != "Qwen/Qwen-Image":
    raise RuntimeError(f"Unexpected Krea 2 VAE default: {verification['vae_default']}")
if verification["vae_scale_factor"] != 8:
    raise RuntimeError(f"Unexpected Krea 2 VAE scale factor: {verification['vae_scale_factor']}")
if not verification["has_save_conversion"] or not verification["has_load_conversion"]:
    raise RuntimeError("The installed Krea 2 implementation lacks required LoRA conversion hooks.")
if not verification["cuda_available"] or not verification["bf16_supported"]:
    raise RuntimeError("CUDA or BF16 is unavailable inside the isolated environment.")

verification["ai_toolkit_commit"] = source_environment_manifest["resolved_commit"]
verification_path = PATHS["config"] / "krea2_implementation_verification.json"
verification_path.write_text(json.dumps(verification, indent=2), encoding="utf-8")
print(json.dumps(verification, indent=2))
print(f"Verification manifest: {verification_path}")


## Cell 5 — Authenticate with Hugging Face

**Code cell.** Requests a read token without writing it into notebook output or project manifests. The token is passed to later subprocesses through the runtime environment.

In [ ]:
import getpass
import json
import os
import subprocess

existing_token = os.environ.get("HF_TOKEN", "").strip()
if existing_token:
    token = existing_token
else:
    token = getpass.getpass("Enter a Hugging Face read token: ").strip()

if not token:
    raise RuntimeError("A Hugging Face token is required to access and download the configured assets.")

os.environ["HF_TOKEN"] = token
validation_script = r"""
import json
import os
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
identity = api.whoami()
print(json.dumps({"authenticated": True, "name": identity.get("name"), "type": identity.get("type")}))
"""
result = subprocess.run(
    [str(PATHS["venv_python"]), "-c", validation_script],
    cwd=str(PATHS["ai_toolkit"]),
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Hugging Face authentication failed.\n{result.stdout}\n{result.stderr}")
record = json.loads(result.stdout.strip().splitlines()[-1])
record_path = PATHS["config"] / "huggingface_authentication.json"
record_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))

## Cell 6 — Resolve and download training assets

**Code cell.** Resolves exact Hugging Face revisions for the Raw checkpoint, Qwen3-VL text encoder, and Qwen-Image VAE. It downloads only the Raw weight file, the full text encoder repository, and the VAE subfolder required by Krea 2. Resolved revisions are saved for reproducibility.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

asset_manifest_path = PATHS["config"] / "training_asset_manifest.json"
minimum_free_bytes = 42 * 1024 ** 3
free_bytes = shutil.disk_usage(PROJECT_ROOT).free
if free_bytes < minimum_free_bytes:
    raise RuntimeError(
        f"At least {minimum_free_bytes / (1024 ** 3):.0f} GiB of free disk is required before downloading training assets. "
        f"Available: {free_bytes / (1024 ** 3):.2f} GiB."
    )

raw_directory = PATHS["models"] / "krea_2_raw"
text_directory = PATHS["models"] / "qwen3_vl_text_encoder"
vae_directory = PATHS["models"] / "qwen_image_vae"

script = r"""
import hashlib
import json
import os
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

config = json.loads(Path(os.environ["KREA2_USER_CONFIG"]).read_text(encoding="utf-8"))
raw_directory = Path(os.environ["KREA2_RAW_DIRECTORY"])
text_directory = Path(os.environ["KREA2_TEXT_DIRECTORY"])
vae_directory = Path(os.environ["KREA2_VAE_DIRECTORY"])
token = os.environ["HF_TOKEN"]
api = HfApi(token=token)

def resolve(repo_id, revision):
    information = api.model_info(repo_id=repo_id, revision=revision or "main", files_metadata=True)
    return information.sha

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

raw_revision = resolve(config["training_model_repository"], config["training_model_revision"])
text_revision = resolve(config["text_encoder_repository"], config["text_encoder_revision"])
vae_revision = resolve(config["vae_repository"], config["vae_revision"])

raw_path = Path(hf_hub_download(
    repo_id=config["training_model_repository"],
    filename=config["training_checkpoint_filename"],
    revision=raw_revision,
    local_dir=str(raw_directory),
    token=token,
))

snapshot_download(
    repo_id=config["text_encoder_repository"],
    revision=text_revision,
    local_dir=str(text_directory),
    token=token,
)

snapshot_download(
    repo_id=config["vae_repository"],
    revision=vae_revision,
    local_dir=str(vae_directory),
    allow_patterns=["vae/*"],
    token=token,
)

result = {
    "training_model": {
        "repository": config["training_model_repository"],
        "revision": raw_revision,
        "checkpoint_path": str(raw_path),
        "checkpoint_filename": raw_path.name,
        "size_bytes": raw_path.stat().st_size,
        "sha256": sha256_file(raw_path),
    },
    "text_encoder": {
        "repository": config["text_encoder_repository"],
        "revision": text_revision,
        "local_directory": str(text_directory),
    },
    "vae": {
        "repository": config["vae_repository"],
        "revision": vae_revision,
        "local_directory": str(vae_directory),
        "subfolder": "vae",
    },
}
print(json.dumps(result))
"""

environment = os.environ.copy()
environment["KREA2_USER_CONFIG"] = str(PATHS["config"] / "user_configuration.json")
environment["KREA2_RAW_DIRECTORY"] = str(raw_directory)
environment["KREA2_TEXT_DIRECTORY"] = str(text_directory)
environment["KREA2_VAE_DIRECTORY"] = str(vae_directory)
environment["PYTHONUNBUFFERED"] = "1"
result = subprocess.run(
    [str(PATHS["venv_python"]), "-c", script],
    cwd=str(PATHS["ai_toolkit"]),
    env=environment,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Training asset download failed.\n{result.stdout}\n{result.stderr}")
manifest = json.loads(result.stdout.strip().splitlines()[-1])
asset_manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print(f"Training asset manifest: {asset_manifest_path}")

## Cell 7 — Upload a dataset ZIP

**Code cell.** Upload exactly one ZIP archive containing matching image-caption files such as `image01.png` and `image01.txt`. Nested folders are accepted; the next cell creates a canonical flat training directory.

In [ ]:
import json
import shutil
from pathlib import Path
from google.colab import files

upload_directory = PATHS["dataset"] / "uploads"
if upload_directory.exists():
    shutil.rmtree(upload_directory)
upload_directory.mkdir(parents=True, exist_ok=False)

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(uploaded) != 1 or len(zip_names) != 1:
    raise RuntimeError("Upload exactly one ZIP archive and no additional files.")

archive_name = zip_names[0]
archive_path = upload_directory / archive_name
archive_path.write_bytes(uploaded[archive_name])
record = {
    "archive_name": archive_name,
    "archive_path": str(archive_path),
    "size_bytes": archive_path.stat().st_size,
}
record_path = PATHS["dataset_audit"] / "dataset_upload_manifest.json"
record_path.parent.mkdir(parents=True, exist_ok=True)
record_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))

## Cell 8 — Extract and validate matching pairs

**Code cell.** Extracts the ZIP safely, rejects unsupported image formats, validates image readability and non-empty captions, rejects duplicate stems, optionally enforces an expected pair count, and copies the validated pairs into a canonical training directory.

In [ ]:
import json
import os
import shutil
import zipfile
from pathlib import Path
from PIL import Image

upload_manifest = json.loads((PATHS["dataset_audit"] / "dataset_upload_manifest.json").read_text(encoding="utf-8"))
archive_path = Path(upload_manifest["archive_path"])
raw_directory = PATHS["dataset_raw"]
training_directory = PATHS["dataset_training"]

for directory in [raw_directory, training_directory]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=False)

raw_root = raw_directory.resolve()
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        destination = (raw_directory / member.filename).resolve()
        try:
            destination.relative_to(raw_root)
        except ValueError as error:
            raise RuntimeError(f"Unsafe ZIP path detected: {member.filename}") from error
    archive.extractall(raw_directory)

allowed_extensions = {".png", ".jpg", ".jpeg"}
unsupported_extensions = {".webp", ".bmp", ".tif", ".tiff", ".gif"}
all_files = [path for path in raw_directory.rglob("*") if path.is_file()]
unsupported_images = [path for path in all_files if path.suffix.lower() in unsupported_extensions]
if unsupported_images:
    raise RuntimeError("Unsupported training image formats were found: " + ", ".join(str(path) for path in unsupported_images))

images = [path for path in all_files if path.suffix.lower() in allowed_extensions]
captions = [path for path in all_files if path.suffix.lower() == ".txt"]

image_map = {}
caption_map = {}
for path in images:
    relative_key = path.relative_to(raw_directory).with_suffix("").as_posix()
    if relative_key in image_map:
        raise RuntimeError(f"Duplicate image key detected: {relative_key}")
    image_map[relative_key] = path
for path in captions:
    relative_key = path.relative_to(raw_directory).with_suffix("").as_posix()
    if relative_key in caption_map:
        raise RuntimeError(f"Duplicate caption key detected: {relative_key}")
    caption_map[relative_key] = path

missing_captions = sorted(set(image_map) - set(caption_map))
missing_images = sorted(set(caption_map) - set(image_map))
if missing_captions or missing_images:
    raise RuntimeError(f"Unmatched files detected. Missing captions: {missing_captions}. Missing images: {missing_images}.")

pair_count = len(image_map)
if pair_count < USER_CONFIG["minimum_pair_count"]:
    raise RuntimeError(f"Dataset contains {pair_count} pairs, below the configured minimum of {USER_CONFIG['minimum_pair_count']}.")
if USER_CONFIG["expected_pair_count"] is not None and pair_count != USER_CONFIG["expected_pair_count"]:
    raise RuntimeError(f"Dataset contains {pair_count} pairs, expected {USER_CONFIG['expected_pair_count']}.")

records = []
for index, source_key in enumerate(sorted(image_map), start=1):
    source_image = image_map[source_key]
    source_caption = caption_map[source_key]
    with Image.open(source_image) as image:
        image.verify()
    with Image.open(source_image) as image:
        width, height = image.size
        mode = image.mode
    caption = source_caption.read_text(encoding="utf-8").strip()
    if not caption:
        raise RuntimeError(f"Caption is empty: {source_caption}")
    canonical_id = f"{index:06d}"
    normalized_extension = source_image.suffix.lower()
    target_image = training_directory / f"{canonical_id}{normalized_extension}"
    target_caption = training_directory / f"{canonical_id}.txt"
    shutil.copy2(source_image, target_image)
    target_caption.write_text(caption, encoding="utf-8")
    records.append({
        "index": index,
        "canonical_id": canonical_id,
        "source_key": source_key,
        "source_image": str(source_image),
        "source_caption": str(source_caption),
        "image": str(target_image),
        "caption": str(target_caption),
        "width": width,
        "height": height,
        "aspect_ratio": width / height,
        "mode": mode,
        "text": caption,
    })

manifest = {
    "pair_count": pair_count,
    "expected_pair_count": USER_CONFIG["expected_pair_count"],
    "minimum_pair_count": USER_CONFIG["minimum_pair_count"],
    "raw_directory": str(raw_directory),
    "training_directory": str(training_directory),
    "canonicalization": "sorted relative source key to six-digit flat identifier",
    "records": records,
}
manifest_path = PATHS["dataset_audit"] / "dataset_validation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Validated image-caption pairs: {pair_count}")
print(f"Canonical training directory: {training_directory}")
print(f"Validation manifest: {manifest_path}")

## Cell 9 — Audit the trigger word without silently rewriting captions

**Code cell.** Counts exact case-sensitive trigger occurrences in every caption. By default, missing triggers cause a clear failure. Set `caption_trigger_policy` to `warn` or `ignore` when appropriate. Optional prefixing is disabled by default and creates backups before changing any caption.

In [ ]:
import json
import shutil
from pathlib import Path

manifest_path = PATHS["dataset_audit"] / "dataset_validation_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
trigger_word = USER_CONFIG["trigger_word"]
policy = USER_CONFIG["caption_trigger_policy"]
backup_directory = PATHS["dataset"] / "caption_backups"

records = []
missing = []
for item in manifest["records"]:
    caption_path = Path(item["caption"])
    text = caption_path.read_text(encoding="utf-8").strip()
    count = text.count(trigger_word) if trigger_word else 0
    if trigger_word and count == 0:
        missing.append(caption_path)
    records.append({"caption": str(caption_path), "trigger_count": count, "text": text})

modified = []
if trigger_word and missing and USER_CONFIG["auto_prefix_missing_trigger"]:
    if backup_directory.exists():
        shutil.rmtree(backup_directory)
    backup_directory.mkdir(parents=True, exist_ok=False)
    for caption_path in missing:
        backup_path = backup_directory / caption_path.name
        shutil.copy2(caption_path, backup_path)
        original = caption_path.read_text(encoding="utf-8").strip()
        caption_path.write_text(f"{trigger_word}, {original}", encoding="utf-8")
        modified.append(str(caption_path))
    missing = []
    records = []
    for item in manifest["records"]:
        caption_path = Path(item["caption"])
        text = caption_path.read_text(encoding="utf-8").strip()
        records.append({"caption": str(caption_path), "trigger_count": text.count(trigger_word), "text": text})

if trigger_word and missing and policy == "require":
    raise RuntimeError("The trigger word is missing from captions: " + ", ".join(path.name for path in missing))

status = "passed"
if trigger_word and missing and policy == "warn":
    status = "warning"

result = {
    "trigger_word": trigger_word,
    "policy": policy,
    "auto_prefix_missing_trigger": USER_CONFIG["auto_prefix_missing_trigger"],
    "modified_captions": modified,
    "missing_caption_count": len(missing),
    "status": status,
    "records": records,
}
result_path = PATHS["dataset_audit"] / "caption_trigger_audit.json"
result_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({key: value for key, value in result.items() if key != "records"}, indent=2, ensure_ascii=False))
print(f"Caption audit: {result_path}")

## Cell 10 — Fingerprint and summarize the dataset

**Code cell.** Creates a deterministic dataset fingerprint from every image and caption, records dimensions and aspect ratios, and saves aggregate statistics. Any later file change produces a different fingerprint.

In [ ]:
import hashlib
import json
import statistics
from pathlib import Path

validation = json.loads((PATHS["dataset_audit"] / "dataset_validation_manifest.json").read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

fingerprint_digest = hashlib.sha256()
records = []
for item in validation["records"]:
    image_path = Path(item["image"])
    caption_path = Path(item["caption"])
    image_hash = sha256_file(image_path)
    caption_hash = sha256_file(caption_path)
    fingerprint_digest.update(item["source_key"].encode("utf-8"))
    fingerprint_digest.update(image_hash.encode("ascii"))
    fingerprint_digest.update(caption_hash.encode("ascii"))
    records.append({
        "canonical_id": item["canonical_id"],
        "source_key": item["source_key"],
        "image_sha256": image_hash,
        "caption_sha256": caption_hash,
        "width": item["width"],
        "height": item["height"],
        "aspect_ratio": item["aspect_ratio"],
    })

widths = [record["width"] for record in records]
heights = [record["height"] for record in records]
ratios = [record["aspect_ratio"] for record in records]
result = {
    "dataset_fingerprint_sha256": fingerprint_digest.hexdigest(),
    "pair_count": len(records),
    "width": {"minimum": min(widths), "maximum": max(widths), "median": statistics.median(widths)},
    "height": {"minimum": min(heights), "maximum": max(heights), "median": statistics.median(heights)},
    "aspect_ratio": {"minimum": min(ratios), "maximum": max(ratios), "median": statistics.median(ratios)},
    "records": records,
}
result_path = PATHS["dataset_audit"] / "dataset_fingerprint.json"
result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(json.dumps({key: value for key, value in result.items() if key != "records"}, indent=2))
print(f"Dataset fingerprint: {result_path}")

## Cell 11 — Visualize every image and caption

**Code cell.** Creates paginated inspection sheets containing every image, filename, dimensions, aspect ratio, and caption. Full-resolution training files are not modified.

In [ ]:
import json
import math
import shutil
import textwrap
from pathlib import Path
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

validation = json.loads((PATHS["dataset_audit"] / "dataset_validation_manifest.json").read_text(encoding="utf-8"))
inspection_directory = PATHS["dataset_audit"] / "inspection_pages"
if inspection_directory.exists():
    shutil.rmtree(inspection_directory)
inspection_directory.mkdir(parents=True, exist_ok=False)

font_path = Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
font = ImageFont.truetype(str(font_path), 18) if font_path.is_file() else ImageFont.load_default()
small_font = ImageFont.truetype(str(font_path), 15) if font_path.is_file() else ImageFont.load_default()
columns = 3
rows = 2
items_per_page = columns * rows
cell_width = 520
cell_height = 670
image_box = 460
page_records = []

for page_index in range(math.ceil(len(validation["records"]) / items_per_page)):
    page_items = validation["records"][page_index * items_per_page:(page_index + 1) * items_per_page]
    canvas = Image.new("RGB", (columns * cell_width, rows * cell_height), "white")
    draw = ImageDraw.Draw(canvas)
    for local_index, item in enumerate(page_items):
        column = local_index % columns
        row = local_index // columns
        x = column * cell_width
        y = row * cell_height
        image = Image.open(item["image"]).convert("RGB")
        image.thumbnail((image_box, image_box), Image.Resampling.LANCZOS)
        image_x = x + (cell_width - image.width) // 2
        image_y = y + 12
        canvas.paste(image, (image_x, image_y))
        information_y = y + image_box + 24
        draw.text((x + 18, information_y), f"{Path(item['image']).name} | {item['source_key']}", fill="black", font=font)
        draw.text((x + 18, information_y + 27), f"{item['width']} x {item['height']} | ratio {item['aspect_ratio']:.3f}", fill="black", font=small_font)
        caption_lines = textwrap.wrap(item["text"], width=58)[:6]
        for line_index, line in enumerate(caption_lines):
            draw.text((x + 18, information_y + 52 + line_index * 19), line, fill="black", font=small_font)
    page_path = inspection_directory / f"inspection_page_{page_index + 1:03d}.png"
    canvas.save(page_path)
    page_records.append(str(page_path))
    display(canvas)

manifest = {"page_count": len(page_records), "items_per_page": items_per_page, "pages": page_records}
manifest_path = PATHS["dataset_audit"] / "inspection_pages_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Inspection pages: {manifest_path}")

## Cell 12 — Audit exact and perceptual duplicates

**Code cell.** Detects exact file duplicates and near-duplicates using a deterministic difference hash. Exact duplicates can be configured to fail the pipeline; perceptual candidates are reported for human review.

In [ ]:
import hashlib
import itertools
import json
from pathlib import Path
from PIL import Image

validation = json.loads((PATHS["dataset_audit"] / "dataset_validation_manifest.json").read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def difference_hash(path, width=16, height=16):
    image = Image.open(path).convert("L").resize((width + 1, height), Image.Resampling.LANCZOS)
    pixels = list(image.getdata())
    bits = []
    for row in range(height):
        offset = row * (width + 1)
        for column in range(width):
            bits.append(pixels[offset + column] > pixels[offset + column + 1])
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value

def hamming_distance(left, right):
    return (left ^ right).bit_count()

records = []
for item in validation["records"]:
    image_path = Path(item["image"])
    records.append({
        "path": str(image_path),
        "filename": image_path.name,
        "source_key": item["source_key"],
        "sha256": sha256_file(image_path),
        "dhash": difference_hash(image_path),
    })

exact_groups = {}
for record in records:
    exact_groups.setdefault(record["sha256"], []).append(record["source_key"])
exact_duplicates = [group for group in exact_groups.values() if len(group) > 1]

threshold = int(USER_CONFIG["near_duplicate_hamming_threshold"])
near_duplicates = []
for left, right in itertools.combinations(records, 2):
    distance = hamming_distance(left["dhash"], right["dhash"])
    if distance <= threshold:
        near_duplicates.append({"left": left["source_key"], "right": right["source_key"], "distance": distance})

result = {
    "near_duplicate_hamming_threshold": threshold,
    "exact_duplicate_groups": exact_duplicates,
    "near_duplicate_candidates": sorted(near_duplicates, key=lambda item: item["distance"]),
    "exact_duplicate_count": len(exact_duplicates),
    "near_duplicate_candidate_count": len(near_duplicates),
}
result_path = PATHS["dataset_audit"] / "duplicate_audit.json"
result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")

if exact_duplicates and USER_CONFIG["fail_on_exact_duplicates"]:
    raise RuntimeError(f"Exact duplicate image groups were found: {exact_duplicates}")

print(json.dumps(result, indent=2))
print(f"Duplicate audit: {result_path}")

## Cell 13 — Write generic production and smoke-test configurations

**Code cell.** Builds AI Toolkit YAML files entirely from the central settings and the resolved asset manifest. It does not contain fixed dataset counts, fixed checkpoint choices, or project-specific prompts.

In [ ]:
import copy
import json
from pathlib import Path
import yaml

asset_manifest = json.loads((PATHS["config"] / "training_asset_manifest.json").read_text(encoding="utf-8"))
fingerprint = json.loads((PATHS["dataset_audit"] / "dataset_fingerprint.json").read_text(encoding="utf-8"))

training_configuration_path = PATHS["config"] / "train_krea2_lora.yaml"
smoke_configuration_path = PATHS["config"] / "train_krea2_lora_smoke.yaml"

sample_section = {
    "sampler": "flowmatch",
    "sample_every": USER_CONFIG["training_sample_every"],
    "sample_start_step": 0,
    "width": USER_CONFIG["inference_width"],
    "height": USER_CONFIG["inference_height"],
    "prompts": USER_CONFIG["evaluation_prompts"],
    "neg": USER_CONFIG["negative_prompt"],
    "seed": USER_CONFIG["evaluation_seeds"][0],
    "walk_seed": False,
    "guidance_scale": USER_CONFIG["raw_sample_guidance"],
    "sample_steps": USER_CONFIG["raw_sample_steps"],
    "network_multiplier": USER_CONFIG["primary_adapter_scale"],
}

configuration = {
    "job": "extension",
    "config": {
        "name": USER_CONFIG["run_name"],
        "process": [
            {
                "type": "sd_trainer",
                "training_folder": str(PATHS["checkpoints"]),
                "device": "cuda:0",
                "trigger_word": USER_CONFIG["trigger_word"],
                "network": {
                    "type": "lora",
                    "linear": USER_CONFIG["lora_rank"],
                    "linear_alpha": USER_CONFIG["lora_alpha"],
                    "transformer_only": True,
                    "all_layers": False,
                    "layer_offloading": USER_CONFIG["layer_offloading"],
                },
                "save": {
                    "dtype": "float16",
                    "save_every": USER_CONFIG["save_every"],
                    "max_step_saves_to_keep": USER_CONFIG["max_step_saves_to_keep"],
                    "save_format": "safetensors",
                    "push_to_hub": False,
                },
                "datasets": [
                    {
                        "folder_path": str(PATHS["dataset_training"]),
                        "caption_ext": "txt",
                        "resolution": USER_CONFIG["training_resolutions"],
                        "buckets": True,
                        "bucket_tolerance": 16,
                        "num_repeats": USER_CONFIG["dataset_repeats"],
                        "caption_dropout_rate": USER_CONFIG["caption_dropout_rate"],
                        "token_dropout_rate": USER_CONFIG["token_dropout_rate"],
                        "shuffle_tokens": USER_CONFIG["shuffle_tokens"],
                        "keep_tokens": USER_CONFIG["keep_tokens"],
                        "random_crop": False,
                        "random_scale": False,
                        "flip_x": USER_CONFIG["flip_x"],
                        "flip_y": False,
                        "cache_latents": False,
                        "cache_latents_to_disk": USER_CONFIG["cache_latents_to_disk"],
                        "cache_text_embeddings": USER_CONFIG["cache_text_embeddings"],
                        "num_workers": 2,
                        "prefetch_factor": 2,
                    }
                ],
                "train": {
                    "batch_size": USER_CONFIG["batch_size"],
                    "steps": USER_CONFIG["training_steps"],
                    "gradient_accumulation": USER_CONFIG["gradient_accumulation"],
                    "train_unet": True,
                    "train_text_encoder": False,
                    "train_refiner": False,
                    "train_turbo": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "timestep_type": "sigmoid",
                    "optimizer": USER_CONFIG["optimizer"],
                    "optimizer_params": {"weight_decay": USER_CONFIG["weight_decay"]},
                    "lr": USER_CONFIG["learning_rate"],
                    "lr_scheduler": USER_CONFIG["lr_scheduler"],
                    "lr_scheduler_params": {},
                    "max_grad_norm": USER_CONFIG["max_grad_norm"],
                    "loss_target": "noise",
                    "loss_type": "mse",
                    "content_or_style": "balanced",
                    "prompt_dropout_prob": 0.0,
                    "cache_text_embeddings": USER_CONFIG["cache_text_embeddings"],
                    "skip_first_sample": True,
                    "disable_sampling": USER_CONFIG["disable_training_samples"],
                    "merge_network_on_save": False,
                    "dtype": USER_CONFIG["training_dtype"],
                },
                "model": {
                    "name_or_path": str(Path(asset_manifest["training_model"]["checkpoint_path"]).parent),
                    "arch": "krea2",
                    "dtype": USER_CONFIG["training_dtype"],
                    "vae_dtype": USER_CONFIG["training_dtype"],
                    "te_dtype": USER_CONFIG["training_dtype"],
                    "quantize": USER_CONFIG["quantize_transformer"],
                    "quantize_te": USER_CONFIG["quantize_text_encoder"],
                    "low_vram": USER_CONFIG["low_vram"],
                    "layer_offloading": USER_CONFIG["layer_offloading"],
                    "split_model_over_gpus": False,
                    "compile": False,
                    "model_kwargs": {
                        "checkpoint_filename": asset_manifest["training_model"]["checkpoint_filename"],
                        "text_encoder_path": asset_manifest["text_encoder"]["local_directory"],
                        "vae_path": asset_manifest["vae"]["local_directory"],
                        "max_text_length": USER_CONFIG["max_text_length"],
                    },
                },
                "sample": sample_section,
            }
        ],
    },
    "meta": {
        "name": "[name]",
        "version": "1.0",
        "project_name": USER_CONFIG["project_name"],
        "concept_type": USER_CONFIG["concept_type"],
        "trigger_word": USER_CONFIG["trigger_word"],
        "base_model": USER_CONFIG["training_model_repository"],
        "base_model_revision": asset_manifest["training_model"]["revision"],
        "dataset_fingerprint_sha256": fingerprint["dataset_fingerprint_sha256"],
    },
}

smoke_configuration = copy.deepcopy(configuration)
smoke_configuration["config"]["name"] = f"{USER_CONFIG['run_name']}_smoke"
smoke_process = smoke_configuration["config"]["process"][0]
smoke_process["training_folder"] = str(PATHS["smoke_checkpoints"])
smoke_process["save"]["save_every"] = 1
smoke_process["save"]["max_step_saves_to_keep"] = max(3, USER_CONFIG["smoke_test_steps"] + 1)
smoke_process["train"]["steps"] = USER_CONFIG["smoke_test_steps"]
smoke_process["train"]["disable_sampling"] = True
smoke_process["sample"]["sample_every"] = 1000000000

training_configuration_path.write_text(yaml.safe_dump(configuration, sort_keys=False, allow_unicode=True), encoding="utf-8")
smoke_configuration_path.write_text(yaml.safe_dump(smoke_configuration, sort_keys=False, allow_unicode=True), encoding="utf-8")

print(f"Production configuration: {training_configuration_path}")
print(f"Smoke configuration: {smoke_configuration_path}")
print(training_configuration_path.read_text(encoding="utf-8"))

## Cell 14 — Preflight the configuration and source contract

**Code cell.** Validates the YAML schema, dataset contents, source paths, adapter settings, Krea 2 model configuration, target LoRA modules, disabled text-encoder training, and non-merged save behavior before any training starts.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

preflight_script = r"""
import json
import os
from pathlib import Path
import yaml
from toolkit.config_modules import ModelConfig, NetworkConfig
from extensions_built_in.diffusion_models.krea2.krea2 import Krea2Model

configuration_path = Path(os.environ["KREA2_TRAINING_CONFIGURATION"])
configuration = yaml.safe_load(configuration_path.read_text(encoding="utf-8"))
processes = configuration.get("config", {}).get("process", [])
if not isinstance(processes, list) or len(processes) != 1:
    raise RuntimeError("The configuration must contain exactly one process.")
process = processes[0]
if process.get("type") != "sd_trainer":
    raise RuntimeError("The process type must be sd_trainer.")
model_section = process.get("model", {})
network_section = process.get("network", {})
train_section = process.get("train", {})
dataset_section = process.get("datasets", [])
if model_section.get("arch") != "krea2":
    raise RuntimeError("The model architecture must be krea2.")
if network_section.get("type") != "lora":
    raise RuntimeError("This notebook supports LoRA network training only.")
if not network_section.get("transformer_only"):
    raise RuntimeError("Krea 2 LoRA training must target the transformer.")
if train_section.get("train_text_encoder"):
    raise RuntimeError("Krea 2 text-encoder training is not supported by this pipeline.")
if train_section.get("merge_network_on_save"):
    raise RuntimeError("Permanent LoRA merging must remain disabled.")
if not isinstance(dataset_section, list) or len(dataset_section) != 1:
    raise RuntimeError("Exactly one canonical dataset directory is required.")
dataset_directory = Path(dataset_section[0]["folder_path"])
images = [path for path in dataset_directory.iterdir() if path.suffix.lower() in {".png", ".jpg", ".jpeg"}]
captions = list(dataset_directory.glob("*.txt"))
if not images or len(images) != len(captions):
    raise RuntimeError("The canonical dataset directory does not contain matching image-caption pairs.")
model_configuration = ModelConfig(**model_section)
network_configuration = NetworkConfig(**network_section)
probe = Krea2Model(device="cuda:0", model_config=model_configuration, dtype=train_section.get("dtype", "bf16"))
result = {
    "run_name": configuration["config"]["name"],
    "dataset_pairs": len(images),
    "model_arch": probe.arch,
    "target_lora_modules": list(probe.target_lora_modules),
    "network_type": network_configuration.type,
    "network_rank": network_configuration.linear,
    "network_alpha": network_configuration.linear_alpha,
    "training_steps": train_section.get("steps"),
    "train_text_encoder": train_section.get("train_text_encoder"),
    "merge_network_on_save": train_section.get("merge_network_on_save"),
    "checkpoint_filename": model_section.get("model_kwargs", {}).get("checkpoint_filename"),
    "text_encoder_path": model_section.get("model_kwargs", {}).get("text_encoder_path"),
    "vae_path": model_section.get("model_kwargs", {}).get("vae_path"),
}
print(json.dumps(result))
"""

environment = os.environ.copy()
environment["KREA2_TRAINING_CONFIGURATION"] = str(PATHS["config"] / "train_krea2_lora.yaml")
result = subprocess.run(
    [str(PATHS["venv_python"]), "-c", preflight_script],
    cwd=str(PATHS["ai_toolkit"]),
    env=environment,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Training preflight failed.\n{result.stdout}\n{result.stderr}")
preflight = json.loads(result.stdout.strip().splitlines()[-1])
if preflight["target_lora_modules"] != ["SingleStreamDiT"]:
    raise RuntimeError(f"Unexpected Krea 2 LoRA target modules: {preflight['target_lora_modules']}")
preflight_path = PATHS["config"] / "training_preflight.json"
preflight_path.write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print(json.dumps(preflight, indent=2))
print(f"Training preflight: {preflight_path}")

## Cell 15 — Run the optional training smoke test

**Code cell.** Runs a very short real training job with the same dataset, model, VAE, text encoder, precision, optimizer, and LoRA architecture as production. Disable it in the central configuration when it has already passed for the same environment.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

if not USER_CONFIG["run_training_smoke_test"]:
    print("Training smoke test skipped by configuration.")
else:
    configuration_path = PATHS["config"] / "train_krea2_lora_smoke.yaml"
    log_path = PATHS["logs"] / "training_smoke_test.log"
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    environment["TOKENIZERS_PARALLELISM"] = "false"
    command = [str(PATHS["venv_python"]), "run.py", str(configuration_path)]
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=str(PATHS["ai_toolkit"]),
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if process.stdout is None:
            raise RuntimeError("Unable to capture the smoke-test process output.")
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
            log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training smoke test failed with exit code {return_code}. Complete log: {log_path}")
    smoke_run_directory = PATHS["smoke_checkpoints"] / f"{USER_CONFIG['run_name']}_smoke"
    smoke_checkpoints = sorted(smoke_run_directory.glob("*.safetensors")) if smoke_run_directory.is_dir() else []
    if not smoke_checkpoints:
        raise RuntimeError("The smoke test completed without producing a LoRA checkpoint.")
    record = {
        "status": "passed",
        "run_directory": str(smoke_run_directory),
        "checkpoint_count": len(smoke_checkpoints),
        "checkpoints": [str(path) for path in smoke_checkpoints],
        "log": str(log_path),
    }
    record_path = PATHS["config"] / "training_smoke_test_result.json"
    record_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(json.dumps(record, indent=2))

## Cell 16 — Start or resume production training

**Code cell.** Starts the configured production run. AI Toolkit resumes from its saved optimizer and checkpoint state when supported by the active source revision. Pressing interrupt is handled as an intentional partial run; the next cell inventories whatever was safely saved. Do not interrupt while a checkpoint is being written.

In [ ]:
import json
import os
import signal
import subprocess
from datetime import datetime, timezone
from pathlib import Path

status_path = PATHS["config"] / "production_training_status.json"
log_path = PATHS["logs"] / "production_training.log"

if not USER_CONFIG["run_production_training"]:
    status = {
        "status": "skipped",
        "reason": "run_production_training is false",
        "training_complete": False,
        "log": str(log_path),
    }
    status_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    print(json.dumps(status, indent=2))
else:
    configuration_path = PATHS["config"] / "train_krea2_lora.yaml"
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    environment["TOKENIZERS_PARALLELISM"] = "false"
    command = [str(PATHS["venv_python"]), "run.py", str(configuration_path)]
    interrupted = False
    with log_path.open("a", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            cwd=str(PATHS["ai_toolkit"]),
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        try:
            if process.stdout is None:
                raise RuntimeError("Unable to capture the production-training process output.")
            for line in process.stdout:
                print(line, end="")
                log_handle.write(line)
                log_handle.flush()
        except KeyboardInterrupt:
            interrupted = True
            process.send_signal(signal.SIGINT)
            process.wait()
            print("Production training was intentionally interrupted. Saved checkpoints will be inventoried next.")
        return_code = process.wait()
    status = {
        "status": "interrupted" if interrupted else "completed_process",
        "process_return_code": return_code,
        "training_complete": False,
        "started_configuration": str(configuration_path),
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "log": str(log_path),
    }
    status_path.write_text(json.dumps(status, indent=2), encoding="utf-8")
    if return_code not in {0, 130, -2} and not interrupted:
        raise RuntimeError(f"Production training failed with exit code {return_code}. Complete log: {log_path}")
    print(json.dumps(status, indent=2))

## Cell 17 — Inventory checkpoints and select an active checkpoint

**Code cell.** Works with completed production runs, intentionally interrupted production runs, or smoke-test-only runs. It discovers actual checkpoint files, validates tensor finiteness and schema consistency, and never requires a predetermined checkpoint list.

Completion is claimed conservatively. A final-looking filename alone is not enough after an interrupted or unknown process state. When numbered checkpoints exist, an untrusted unnumbered final file is excluded from automatic selection unless process completion or the configured final step is independently confirmed.


In [ ]:
import json
import os
import re
import subprocess
from pathlib import Path

production_directory = PATHS["checkpoints"] / USER_CONFIG["run_name"]
smoke_directory = PATHS["smoke_checkpoints"] / f"{USER_CONFIG['run_name']}_smoke"
production_status_path = PATHS["config"] / "production_training_status.json"
smoke_status_path = PATHS["config"] / "training_smoke_test_result.json"

if production_directory.is_dir() and list(production_directory.glob("*.safetensors")):
    selected_run_directory = production_directory
    selected_run_name = USER_CONFIG["run_name"]
    source_kind = "production"
    process_status = (
        json.loads(production_status_path.read_text(encoding="utf-8"))
        if production_status_path.is_file()
        else {}
    )
elif smoke_directory.is_dir() and list(smoke_directory.glob("*.safetensors")):
    selected_run_directory = smoke_directory
    selected_run_name = f"{USER_CONFIG['run_name']}_smoke"
    source_kind = "smoke"
    process_status = (
        json.loads(smoke_status_path.read_text(encoding="utf-8"))
        if smoke_status_path.is_file()
        else {}
    )
else:
    raise RuntimeError("No production or smoke-test LoRA checkpoints were found.")

inventory_script = r"""
import hashlib
import json
import os
import re
from pathlib import Path
import torch
from safetensors import safe_open

run_directory = Path(os.environ["KREA2_RUN_DIRECTORY"])
run_name = os.environ["KREA2_RUN_NAME"]
configured_steps = int(os.environ["KREA2_CONFIGURED_STEPS"])
source_kind = os.environ["KREA2_SOURCE_KIND"]
process_status = json.loads(os.environ["KREA2_PROCESS_STATUS_JSON"])


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


records = []
reference_keys = None
reference_shapes = None
for path in sorted(run_directory.glob("*.safetensors")):
    final_name = path.stem == run_name
    match = re.fullmatch(re.escape(run_name) + r"_(\d+)", path.stem)
    if not final_name and match is None:
        continue
    step = configured_steps if final_name else int(match.group(1))
    tensor_count = 0
    parameter_count = 0
    nonfinite_count = 0
    ranks = set()
    keys = []
    shapes = {}
    metadata = {}
    with safe_open(path, framework="pt", device="cpu") as checkpoint:
        metadata = checkpoint.metadata() or {}
        keys = list(checkpoint.keys())
        for key in keys:
            tensor = checkpoint.get_tensor(key)
            shapes[key] = list(tensor.shape)
            tensor_count += 1
            parameter_count += int(tensor.numel())
            nonfinite_count += int((~torch.isfinite(tensor.float())).sum().item())
            if key.endswith(".lora_A.weight") and tensor.ndim == 2:
                ranks.add(int(tensor.shape[0]))
    if nonfinite_count != 0:
        raise RuntimeError(f"Checkpoint contains non-finite parameters: {path}")
    if len(ranks) != 1:
        raise RuntimeError(f"Unable to infer one LoRA rank from {path}: {sorted(ranks)}")
    if reference_keys is None:
        reference_keys = keys
        reference_shapes = shapes
    else:
        if keys != reference_keys or shapes != reference_shapes:
            raise RuntimeError(f"Checkpoint schema differs from the first checkpoint: {path}")
    records.append({
        "step": step,
        "path": str(path),
        "filename": path.name,
        "is_final_name": final_name,
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
        "tensor_count": tensor_count,
        "parameter_count": parameter_count,
        "nonfinite_parameter_count": nonfinite_count,
        "rank": next(iter(ranks)),
        "metadata": metadata,
    })

if not records:
    raise RuntimeError("No valid LoRA checkpoint files were discovered.")

records.sort(key=lambda item: (item["step"], item["is_final_name"]))
final_records = [record for record in records if record["is_final_name"]]
numbered_records = [record for record in records if not record["is_final_name"]]
latest_numbered_step = max((record["step"] for record in numbered_records), default=None)

production_process_completed = (
    source_kind == "production"
    and process_status.get("status") == "completed_process"
    and process_status.get("process_return_code") == 0
)
smoke_process_completed = (
    source_kind == "smoke"
    and process_status.get("status") == "passed"
)
configured_step_reached = latest_numbered_step is not None and latest_numbered_step >= configured_steps
training_complete = bool(
    configured_step_reached
    or (production_process_completed and final_records)
    or smoke_process_completed
)

completion_evidence = []
if configured_step_reached:
    completion_evidence.append("numbered_checkpoint_reached_configured_step")
if production_process_completed and final_records:
    completion_evidence.append("production_process_returned_zero_with_final_checkpoint")
if smoke_process_completed:
    completion_evidence.append("smoke_process_passed")
if not completion_evidence:
    completion_evidence.append("no_independent_completion_evidence")

excluded_untrusted_final_files = []
trusted_records = records
if final_records and not training_complete and numbered_records:
    trusted_records = numbered_records
    excluded_untrusted_final_files = [record["path"] for record in final_records]

step_map = {}
for record in trusted_records:
    step_map[record["step"]] = record
unique_records = [step_map[step] for step in sorted(step_map)]

if not unique_records:
    raise RuntimeError("No trusted checkpoint records remain after completion-state validation.")

optimizer_files = [path for path in run_directory.rglob("optimizer.pt") if path.is_file()]
result = {
    "source_kind": source_kind,
    "run_name": run_name,
    "run_directory": str(run_directory),
    "configured_steps": configured_steps,
    "training_complete": training_complete,
    "completion_evidence": completion_evidence,
    "process_status": process_status,
    "checkpoint_count": len(unique_records),
    "checkpoint_steps": [record["step"] for record in unique_records],
    "checkpoints": unique_records,
    "excluded_untrusted_final_files": excluded_untrusted_final_files,
    "optimizer_state_present": bool(optimizer_files),
    "optimizer_files": [str(path) for path in optimizer_files],
    "final_quality_claim_allowed": source_kind == "production" and training_complete,
}
print(json.dumps(result))
"""

environment = os.environ.copy()
environment["KREA2_RUN_DIRECTORY"] = str(selected_run_directory)
environment["KREA2_RUN_NAME"] = selected_run_name
environment["KREA2_CONFIGURED_STEPS"] = str(
    USER_CONFIG["training_steps"] if source_kind == "production" else USER_CONFIG["smoke_test_steps"]
)
environment["KREA2_SOURCE_KIND"] = source_kind
environment["KREA2_PROCESS_STATUS_JSON"] = json.dumps(process_status)
result = subprocess.run(
    [str(PATHS["venv_python"]), "-c", inventory_script],
    cwd=str(PATHS["ai_toolkit"]),
    env=environment,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Checkpoint inventory failed.\n{result.stdout}\n{result.stderr}")
inventory = json.loads(result.stdout.strip().splitlines()[-1])

mode = USER_CONFIG["checkpoint_selection_mode"]
if mode == "manual":
    step = USER_CONFIG["manual_checkpoint_step"]
    if step is None:
        raise RuntimeError("manual_checkpoint_step must be set when checkpoint_selection_mode is manual.")
    candidates = [record for record in inventory["checkpoints"] if record["step"] == step]
    if len(candidates) != 1:
        raise RuntimeError(f"Manual checkpoint step {step} was not found exactly once.")
    selected = candidates[0]
elif mode == "latest":
    selected = inventory["checkpoints"][-1]
else:
    final_candidates = [
        record
        for record in inventory["checkpoints"]
        if record["is_final_name"] and inventory["training_complete"]
    ]
    selected = final_candidates[-1] if final_candidates else inventory["checkpoints"][-1]

inventory_path = PATHS["config"] / "checkpoint_inventory.json"
selection_path = PATHS["config"] / "active_checkpoint_selection.json"
inventory_path.write_text(json.dumps(inventory, indent=2, ensure_ascii=False), encoding="utf-8")
selection = {
    "selection_mode": mode,
    "source_kind": inventory["source_kind"],
    "training_complete": inventory["training_complete"],
    "completion_evidence": inventory["completion_evidence"],
    "final_quality_claim_allowed": inventory["final_quality_claim_allowed"],
    "checkpoint_step": selected["step"],
    "checkpoint_path": selected["path"],
    "checkpoint_sha256": selected["sha256"],
    "rank": selected["rank"],
    "adapter_name": USER_CONFIG["primary_adapter_name"],
    "adapter_scale": USER_CONFIG["primary_adapter_scale"],
    "pipeline_smoke_test_only": not inventory["final_quality_claim_allowed"],
    "permanent_merge_allowed": False,
}
selection_path.write_text(json.dumps(selection, indent=2), encoding="utf-8")
print(json.dumps({key: value for key, value in inventory.items() if key != "checkpoints"}, indent=2))
print(json.dumps(selection, indent=2))
print(f"Checkpoint inventory: {inventory_path}")
print(f"Active checkpoint selection: {selection_path}")


## Cell 18 — Validate optional additional LoRA declarations

**Code cell.** Checks names, paths, scales, ranks, and duplicate declarations for optional compatible Krea 2 LoRAs. Compatibility is not inferred from filenames; every additional adapter must be explicitly supplied and successfully loaded by the inference cells.

In [ ]:
import json
from pathlib import Path

records = []
names = {USER_CONFIG["primary_adapter_name"]}
for item in USER_CONFIG["additional_loras"]:
    required = {"name", "path", "scale"}
    missing = required - set(item)
    if missing:
        raise RuntimeError(f"Additional LoRA declaration is missing fields: {sorted(missing)}")
    if item["name"] in names:
        raise RuntimeError(f"Duplicate adapter name: {item['name']}")
    names.add(item["name"])
    path = Path(item["path"])
    if not path.is_file():
        raise RuntimeError(f"Additional LoRA file is missing: {path}")
    records.append({
        "name": item["name"],
        "path": str(path),
        "scale": float(item["scale"]),
        "alpha": item.get("alpha"),
        "exists": True,
    })
result = {"additional_lora_count": len(records), "records": records, "permanent_merge_allowed": False}
result_path = PATHS["config"] / "additional_lora_validation.json"
result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(json.dumps(result, indent=2))

## Cell 19 — Resolve and download the inference checkpoint

**Code cell.** Downloads Krea-2-Turbo only when inference is enabled. The official Turbo contract uses eight denoising steps and guidance `0.0`; both remain editable but are validated against the configured model family.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

if not USER_CONFIG["run_inference"]:
    print("Inference asset preparation skipped by configuration.")
else:
    training_assets = json.loads((PATHS["config"] / "training_asset_manifest.json").read_text(encoding="utf-8"))
    turbo_directory = PATHS["models"] / "krea_2_turbo"
    manifest_path = PATHS["config"] / "inference_asset_manifest.json"
    free_bytes = shutil.disk_usage(PROJECT_ROOT).free
    if free_bytes < 27 * 1024 ** 3 and not (turbo_directory / USER_CONFIG["inference_checkpoint_filename"]).is_file():
        raise RuntimeError(f"Insufficient free disk for Krea-2-Turbo. Available: {free_bytes / (1024 ** 3):.2f} GiB.")
    script = r"""
import hashlib
import json
import os
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
config = json.loads(Path(os.environ["KREA2_USER_CONFIG"]).read_text(encoding="utf-8"))
turbo_directory = Path(os.environ["KREA2_TURBO_DIRECTORY"])
token = os.environ["HF_TOKEN"]
api = HfApi(token=token)
information = api.model_info(
    repo_id=config["inference_model_repository"],
    revision=config["inference_model_revision"] or "main",
    files_metadata=True,
)
revision = information.sha
path = Path(hf_hub_download(
    repo_id=config["inference_model_repository"],
    filename=config["inference_checkpoint_filename"],
    revision=revision,
    local_dir=str(turbo_directory),
    token=token,
))
digest = hashlib.sha256()
with path.open("rb") as handle:
    for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
        digest.update(block)
print(json.dumps({
    "repository": config["inference_model_repository"],
    "revision": revision,
    "checkpoint_path": str(path),
    "checkpoint_filename": path.name,
    "size_bytes": path.stat().st_size,
    "sha256": digest.hexdigest(),
}))
"""
    environment = os.environ.copy()
    environment["KREA2_USER_CONFIG"] = str(PATHS["config"] / "user_configuration.json")
    environment["KREA2_TURBO_DIRECTORY"] = str(turbo_directory)
    result = subprocess.run(
        [str(PATHS["venv_python"]), "-c", script],
        cwd=str(PATHS["ai_toolkit"]),
        env=environment,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Inference asset preparation failed.\n{result.stdout}\n{result.stderr}")
    turbo = json.loads(result.stdout.strip().splitlines()[-1])
    manifest = {
        "inference_model": turbo,
        "text_encoder": training_assets["text_encoder"],
        "vae": training_assets["vae"],
        "official_turbo_defaults": {"num_inference_steps": 8, "guidance_scale": 0.0},
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(json.dumps(manifest, indent=2))

## Cell 20 — Write non-destructive Krea 2 inference helpers

**Code cell.** Writes a reusable helper module inside the project. It loads the Turbo transformer once, attaches separately named compatible LoRAs with independent scales, converts AI Toolkit Krea 2 LoRA keys before loading, and permanently disables merge operations.

The live adapter multiplier is validated through each network's actual `torch_multiplier` tensor. The check compares against the value represented in that tensor's runtime dtype, so BF16 rounding is treated correctly instead of being mistaken for a scale failure.


In [ ]:
from pathlib import Path

helper_path = PATHS["helpers"] / "krea2_runtime.py"
helper_source = r"""
import gc
import hashlib
from pathlib import Path
import torch
from safetensors import safe_open
from safetensors.torch import load_file
from toolkit.config_modules import ModelConfig, NetworkConfig
from toolkit.lora_special import LoRASpecialNetwork
from extensions_built_in.diffusion_models.krea2.krea2 import Krea2Model


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def infer_lora_rank(path):
    with safe_open(Path(path), framework="pt", device="cpu") as checkpoint:
        key = next((name for name in checkpoint.keys() if name.endswith(".lora_A.weight")), None)
        if key is None:
            raise RuntimeError(f"No LoRA A tensor was found in {path}")
        return int(checkpoint.get_slice(key).get_shape()[0])


def capture_base_parameter_samples(transformer, sample_count=6, sample_length=2048):
    candidates = [
        (name, parameter)
        for name, parameter in transformer.named_parameters()
        if "lora_" not in name.lower() and ".lora" not in name.lower()
    ]
    if not candidates:
        raise RuntimeError("No non-LoRA transformer parameters were found for the non-destructive audit.")
    indices = sorted(set(round(index * (len(candidates) - 1) / max(sample_count - 1, 1)) for index in range(min(sample_count, len(candidates)))))
    return {
        candidates[index][0]: candidates[index][1].detach().reshape(-1)[:sample_length].float().cpu().clone()
        for index in indices
    }


def compare_base_parameter_samples(transformer, samples):
    parameter_map = dict(transformer.named_parameters())
    records = []
    unchanged = True
    for name, before in samples.items():
        if name not in parameter_map:
            raise RuntimeError(f"A sampled base parameter disappeared: {name}")
        after = parameter_map[name].detach().reshape(-1)[:before.numel()].float().cpu()
        equal = bool(torch.equal(before, after))
        unchanged = unchanged and equal
        records.append({
            "name": name,
            "unchanged": equal,
            "maximum_difference": float((before - after).abs().max().item()),
        })
    return unchanged, records


class MultiAdapterController:
    def __init__(self, records):
        self.records = records
        self.can_merge_in = False
        self.is_merged_in = False
        self.is_active = True
        self.training = False
        self._multiplier = 1.0

    @property
    def multiplier(self):
        return self._multiplier

    @multiplier.setter
    def multiplier(self, value):
        self._multiplier = float(value)
        self._apply()

    def _apply(self):
        for record in self.records:
            network = record["network"]
            effective = float(record["scale"]) * self._multiplier
            network.multiplier = effective
            network.is_active = effective != 0.0
            network.can_merge_in = False
            network.is_merged_in = False
            network._update_torch_multiplier()
            multiplier_tensor = network.torch_multiplier.detach()
            if not torch.isfinite(multiplier_tensor.float()).all():
                raise RuntimeError(f"Adapter multiplier is non-finite: {record['name']}")
            expected_value = torch.tensor(
                effective,
                dtype=multiplier_tensor.dtype,
                device=multiplier_tensor.device,
            ).float()
            received_values = multiplier_tensor.float()
            if not torch.all(received_values == expected_value):
                raise RuntimeError(
                    f"Adapter multiplier mismatch for {record['name']}: "
                    f"expected {expected_value.item()}, received {received_values.cpu().tolist()}"
                )

    def set_scales(self, scales):
        unknown = set(scales) - {record["name"] for record in self.records}
        if unknown:
            raise RuntimeError(f"Unknown adapter names: {sorted(unknown)}")
        for record in self.records:
            if record["name"] in scales:
                record["scale"] = float(scales[record["name"]])
        self._apply()

    def set_global_multiplier(self, value):
        self.multiplier = value

    def eval(self):
        self.training = False
        for record in self.records:
            record["network"].eval()
        return self

    def train(self, mode=True):
        self.training = bool(mode)
        for record in self.records:
            record["network"].train(mode)
        return self

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False


def load_runtime(inference_assets, adapter_specs, dtype="bf16", max_text_length=512):
    turbo = inference_assets["inference_model"]
    model_configuration = ModelConfig(
        name_or_path=str(Path(turbo["checkpoint_path"]).parent),
        arch="krea2",
        dtype=dtype,
        vae_dtype=dtype,
        te_dtype=dtype,
        quantize=False,
        quantize_te=False,
        low_vram=False,
        layer_offloading=False,
        split_model_over_gpus=False,
        compile=False,
        assistant_lora_path=None,
        inference_lora_path=None,
        model_kwargs={
            "checkpoint_filename": turbo["checkpoint_filename"],
            "text_encoder_path": inference_assets["text_encoder"]["local_directory"],
            "vae_path": inference_assets["vae"]["local_directory"],
            "max_text_length": max_text_length,
        },
    )
    model = Krea2Model(device="cuda:0", model_config=model_configuration, dtype=dtype)
    model.load_model()
    transformer = getattr(model, "unet", None)
    if transformer is None:
        transformer = model.get_model_to_train()
    if isinstance(transformer, (list, tuple)):
        if len(transformer) != 1:
            raise RuntimeError(f"Unexpected transformer count: {len(transformer)}")
        transformer = transformer[0]
    if transformer is None:
        raise RuntimeError("The Krea 2 transformer could not be resolved.")
    transformer.eval()
    target_modules = list(model.target_lora_modules)
    if target_modules != ["SingleStreamDiT"]:
        raise RuntimeError(f"Unexpected Krea 2 LoRA target modules: {target_modules}")
    LoRASpecialNetwork.LORA_PREFIX_UNET = "lora_transformer"
    records = []
    names = set()
    for adapter_spec in adapter_specs:
        name = adapter_spec["name"]
        if name in names:
            raise RuntimeError(f"Duplicate adapter name: {name}")
        names.add(name)
        adapter_path = Path(adapter_spec["path"])
        if not adapter_path.is_file():
            raise RuntimeError(f"Adapter file is missing: {adapter_path}")
        rank = infer_lora_rank(adapter_path)
        alpha_value = adapter_spec.get("alpha")
        alpha = rank if alpha_value is None else int(alpha_value)
        network_configuration = NetworkConfig(type="lora", linear=rank, linear_alpha=alpha, transformer_only=True)
        network = LoRASpecialNetwork(
            text_encoder=None,
            unet=transformer,
            lora_dim=rank,
            multiplier=float(adapter_spec.get("scale", 1.0)),
            alpha=alpha,
            train_unet=True,
            train_text_encoder=False,
            network_config=network_configuration,
            network_type="lora",
            transformer_only=True,
            is_transformer=True,
            target_lin_modules=target_modules,
            base_model=model,
        )
        network.adapter_name = name
        network.apply_to(None, transformer, apply_text_encoder=False, apply_unet=True)
        network.force_to(model.device_torch, dtype=model.torch_dtype)
        network.eval()
        network.can_merge_in = False
        network.is_merged_in = False
        state_dict = load_file(str(adapter_path))
        state_dict = model.convert_lora_weights_before_load(state_dict)
        network.load_weights(state_dict)
        nonfinite_count = sum(int((~torch.isfinite(parameter.detach().float())).sum().item()) for parameter in network.parameters())
        if nonfinite_count != 0:
            raise RuntimeError(f"Adapter contains non-finite runtime parameters: {adapter_path}")
        records.append({
            "name": name,
            "path": str(adapter_path),
            "sha256": sha256_file(adapter_path),
            "scale": float(adapter_spec.get("scale", 1.0)),
            "rank": rank,
            "alpha": alpha,
            "parameter_count": sum(int(parameter.numel()) for parameter in network.parameters()),
            "network": network,
        })
    controller = MultiAdapterController(records)
    controller.set_scales({record["name"]: record["scale"] for record in records})
    model.network = controller
    return model, transformer, controller, records


def unload_runtime(model, transformer, controller, records):
    controller.set_scales({record["name"]: 0.0 for record in records})
    del records
    del controller
    del transformer
    del model
    gc.collect()
    torch.cuda.empty_cache()
"""
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text(helper_source, encoding="utf-8")
print(f"Inference helper module: {helper_path}")

## Cell 21 — Generate a deterministic base-versus-LoRA smoke comparison

**Code cell.** Uses the currently selected checkpoint, one editable evaluation prompt and seed, and identical Turbo settings for both images. The only change is the active adapter scales. It validates that the LoRA has a measurable effect, base parameters remain unchanged, and no merge occurred. Partial checkpoints are accepted for pipeline testing without quality claims.

In [ ]:
import json
import os
import subprocess
from pathlib import Path
from IPython.display import display
from PIL import Image

if not USER_CONFIG["run_inference"]:
    print("Inference smoke comparison skipped by configuration.")
else:
    output_directory = PATHS["inference"] / "smoke_comparison"
    output_directory.mkdir(parents=True, exist_ok=True)
    script = r"""
import json
import os
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageDraw
from toolkit.config_modules import GenerateImageConfig
from runtime_helpers.krea2_runtime import capture_base_parameter_samples, compare_base_parameter_samples, load_runtime, sha256_file, unload_runtime

root = Path(os.environ["KREA2_PROJECT_ROOT"])
config = json.loads((root / "config" / "user_configuration.json").read_text(encoding="utf-8"))
selection = json.loads((root / "config" / "active_checkpoint_selection.json").read_text(encoding="utf-8"))
assets = json.loads((root / "config" / "inference_asset_manifest.json").read_text(encoding="utf-8"))
output_directory = Path(os.environ["KREA2_OUTPUT_DIRECTORY"])
additional = config["additional_loras"]
adapter_specs = [{
    "name": config["primary_adapter_name"],
    "path": selection["checkpoint_path"],
    "scale": config["primary_adapter_scale"],
    "alpha": config["lora_alpha"],
}] + additional
model, transformer, controller, records = load_runtime(assets, adapter_specs, dtype=config["training_dtype"], max_text_length=config["max_text_length"])
base_samples = capture_base_parameter_samples(transformer)
base_scales = {record["name"]: 0.0 for record in records}
active_scales = {config["primary_adapter_name"]: config["primary_adapter_scale"]}
for item in additional:
    active_scales[item["name"]] = float(item.get("scale", 1.0))
prompt = config["evaluation_prompts"][0]
seed = int(config["evaluation_seeds"][0])
base_path = output_directory / "base.png"
active_path = output_directory / "active_loras.png"
controller.set_scales(base_scales)
model.generate_images([GenerateImageConfig(
    prompt=prompt,
    width=int(config["inference_width"]),
    height=int(config["inference_height"]),
    num_inference_steps=int(config["inference_steps"]),
    guidance_scale=float(config["inference_guidance"]),
    negative_prompt=config["negative_prompt"],
    seed=seed,
    network_multiplier=1.0,
    output_path=str(base_path),
    output_ext="png",
    add_prompt_file=False,
)])
controller.set_scales(active_scales)
model.generate_images([GenerateImageConfig(
    prompt=prompt,
    width=int(config["inference_width"]),
    height=int(config["inference_height"]),
    num_inference_steps=int(config["inference_steps"]),
    guidance_scale=float(config["inference_guidance"]),
    negative_prompt=config["negative_prompt"],
    seed=seed,
    network_multiplier=1.0,
    output_path=str(active_path),
    output_ext="png",
    add_prompt_file=False,
)])
if not base_path.is_file() or not active_path.is_file():
    raise RuntimeError("The smoke comparison images were not created.")
base_image = Image.open(base_path).convert("RGB")
active_image = Image.open(active_path).convert("RGB")
base_array = np.asarray(base_image, dtype=np.float32)
active_array = np.asarray(active_image, dtype=np.float32)
pixel_mae = float(np.mean(np.abs(base_array - active_array)))
if pixel_mae <= 0.0:
    raise RuntimeError("The base and active-adapter outputs are identical.")
label_height = 52
grid = Image.new("RGB", (base_image.width + active_image.width, max(base_image.height, active_image.height) + label_height), "white")
grid.paste(base_image, (0, label_height))
grid.paste(active_image, (base_image.width, label_height))
draw = ImageDraw.Draw(grid)
draw.text((18, 17), "Turbo base", fill="black")
draw.text((base_image.width + 18, 17), "Turbo with active LoRAs", fill="black")
grid_path = output_directory / "base_vs_active_grid.png"
grid.save(grid_path)
base_unchanged, base_parameter_records = compare_base_parameter_samples(transformer, base_samples)
if not base_unchanged:
    raise RuntimeError("Representative Turbo base parameters changed during inference.")
if any(record["network"].can_merge_in or record["network"].is_merged_in for record in records):
    raise RuntimeError("At least one adapter entered a merge-capable or merged state.")
metadata = {
    "checkpoint_step": selection["checkpoint_step"],
    "checkpoint_path": selection["checkpoint_path"],
    "source_kind": selection["source_kind"],
    "training_complete": selection["training_complete"],
    "final_quality_claim_allowed": selection["final_quality_claim_allowed"],
    "prompt": prompt,
    "seed": seed,
    "width": config["inference_width"],
    "height": config["inference_height"],
    "num_inference_steps": config["inference_steps"],
    "guidance_scale": config["inference_guidance"],
    "base_scales": base_scales,
    "active_scales": active_scales,
    "pixel_mae": pixel_mae,
    "base_path": str(base_path),
    "active_path": str(active_path),
    "grid_path": str(grid_path),
    "grid_sha256": sha256_file(grid_path),
    "base_parameters_unchanged": base_unchanged,
    "base_parameter_records": base_parameter_records,
    "permanent_merge_performed": False,
}
metadata_path = output_directory / "smoke_comparison_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("__KREA2_RESULT__" + json.dumps(metadata))
unload_runtime(model, transformer, controller, records)
"""
    environment = os.environ.copy()
    environment["KREA2_PROJECT_ROOT"] = str(PROJECT_ROOT)
    environment["KREA2_OUTPUT_DIRECTORY"] = str(output_directory)
    environment["PYTHONPATH"] = str(PROJECT_ROOT)
    environment["PYTHONUNBUFFERED"] = "1"
    result = subprocess.run(
        [str(PATHS["venv_python"]), "-c", script],
        cwd=str(PATHS["ai_toolkit"]),
        env=environment,
        capture_output=True,
        text=True,
    )
    log_path = PATHS["logs"] / "inference_smoke_comparison.log"
    log_path.write_text(result.stdout + result.stderr + f"\nExit code: {result.returncode}\n", encoding="utf-8")
    if result.returncode != 0:
        raise RuntimeError(f"Inference smoke comparison failed.\n{result.stdout}\n{result.stderr}\nComplete log: {log_path}")
    structured = [line for line in result.stdout.splitlines() if line.startswith("__KREA2_RESULT__")]
    if len(structured) != 1:
        raise RuntimeError("The inference smoke comparison returned an invalid structured result.")
    metadata = json.loads(structured[0].removeprefix("__KREA2_RESULT__"))
    print(json.dumps(metadata, indent=2))
    display(Image.open(metadata["grid_path"]).convert("RGB"))

## Cell 22 — Run a dynamic checkpoint sweep

**Code cell.** Selects checkpoints from the actual inventory. `auto` chooses evenly distributed checkpoints up to the configured maximum; `all`, `manual`, and `selected_only` are also supported. It uses the same prompts, seeds, dimensions, steps, guidance, VAE, and scale for every checkpoint. It creates comparison grids but does not choose a winner.

In [ ]:
import json
import math
import os
import subprocess
from pathlib import Path
from IPython.display import display
from PIL import Image

if not USER_CONFIG["run_inference"] or not USER_CONFIG["run_checkpoint_sweep"]:
    print("Checkpoint sweep skipped by configuration.")
else:
    inventory = json.loads((PATHS["config"] / "checkpoint_inventory.json").read_text(encoding="utf-8"))
    selection = json.loads((PATHS["config"] / "active_checkpoint_selection.json").read_text(encoding="utf-8"))
    checkpoints = inventory["checkpoints"]
    mode = USER_CONFIG["checkpoint_sweep_mode"]
    if mode == "selected_only":
        sweep_records = [next(record for record in checkpoints if record["path"] == selection["checkpoint_path"])]
    elif mode == "manual":
        requested = USER_CONFIG["manual_sweep_steps"]
        sweep_records = [record for record in checkpoints if record["step"] in requested]
        if sorted(record["step"] for record in sweep_records) != sorted(set(requested)):
            raise RuntimeError("At least one manually requested checkpoint step is missing.")
    elif mode == "all" or len(checkpoints) <= USER_CONFIG["maximum_sweep_checkpoints"]:
        sweep_records = checkpoints
    else:
        maximum = USER_CONFIG["maximum_sweep_checkpoints"]
        indices = sorted(set(round(index * (len(checkpoints) - 1) / (maximum - 1)) for index in range(maximum))) if maximum > 1 else [len(checkpoints) - 1]
        sweep_records = [checkpoints[index] for index in indices]
    if not sweep_records:
        raise RuntimeError("The dynamic checkpoint sweep selected no checkpoints.")
    output_directory = PATHS["inference"] / "checkpoint_sweep"
    output_directory.mkdir(parents=True, exist_ok=True)
    sweep_request_path = PATHS["config"] / "checkpoint_sweep_request.json"
    sweep_request = {"checkpoints": sweep_records, "mode": mode}
    sweep_request_path.write_text(json.dumps(sweep_request, indent=2), encoding="utf-8")
    script = r"""
import gc
import json
import os
from pathlib import Path
import torch
from PIL import Image, ImageDraw
from safetensors.torch import load_file
from toolkit.config_modules import GenerateImageConfig
from runtime_helpers.krea2_runtime import load_runtime, sha256_file, unload_runtime

root = Path(os.environ["KREA2_PROJECT_ROOT"])
config = json.loads((root / "config" / "user_configuration.json").read_text(encoding="utf-8"))
assets = json.loads((root / "config" / "inference_asset_manifest.json").read_text(encoding="utf-8"))
request = json.loads((root / "config" / "checkpoint_sweep_request.json").read_text(encoding="utf-8"))
output_directory = Path(os.environ["KREA2_OUTPUT_DIRECTORY"])
first = request["checkpoints"][0]
adapter_specs = [{
    "name": config["primary_adapter_name"],
    "path": first["path"],
    "scale": config["primary_adapter_scale"],
    "alpha": config["lora_alpha"],
}] + config["additional_loras"]
model, transformer, controller, records = load_runtime(assets, adapter_specs, dtype=config["training_dtype"], max_text_length=config["max_text_length"])
primary_record = records[0]
active_scales = {config["primary_adapter_name"]: config["primary_adapter_scale"]}
for item in config["additional_loras"]:
    active_scales[item["name"]] = float(item.get("scale", 1.0))
controller.set_scales(active_scales)
image_records = []
paths_by_case = {index: [] for index in range(len(config["evaluation_prompts"]))}
for checkpoint in request["checkpoints"]:
    state_dict = load_file(checkpoint["path"])
    state_dict = model.convert_lora_weights_before_load(state_dict)
    primary_record["network"].load_weights(state_dict)
    controller.set_scales(active_scales)
    configurations = []
    expected = []
    for case_index, prompt in enumerate(config["evaluation_prompts"]):
        seed = int(config["evaluation_seeds"][case_index % len(config["evaluation_seeds"])])
        output_path = output_directory / f"case_{case_index + 1:02d}_step_{int(checkpoint['step']):08d}.png"
        configurations.append(GenerateImageConfig(
            prompt=prompt,
            width=int(config["inference_width"]),
            height=int(config["inference_height"]),
            num_inference_steps=int(config["inference_steps"]),
            guidance_scale=float(config["inference_guidance"]),
            negative_prompt=config["negative_prompt"],
            seed=seed,
            network_multiplier=1.0,
            output_path=str(output_path),
            output_ext="png",
            add_prompt_file=False,
        ))
        expected.append((case_index, prompt, seed, output_path))
    model.generate_images(configurations)
    for case_index, prompt, seed, output_path in expected:
        if not output_path.is_file():
            raise RuntimeError(f"Expected sweep image is missing: {output_path}")
        image_records.append({
            "case_index": case_index + 1,
            "checkpoint_step": int(checkpoint["step"]),
            "checkpoint_path": checkpoint["path"],
            "prompt": prompt,
            "seed": seed,
            "path": str(output_path),
            "sha256": sha256_file(output_path),
        })
        paths_by_case[case_index].append({"step": int(checkpoint["step"]), "path": str(output_path)})
    gc.collect()
thumbnail = 384
label_height = 48
grid_paths = []
master_rows = []
for case_index in range(len(config["evaluation_prompts"])):
    items = sorted(paths_by_case[case_index], key=lambda item: item["step"])
    grid = Image.new("RGB", (thumbnail * len(items), thumbnail + label_height), "white")
    draw = ImageDraw.Draw(grid)
    for column, item in enumerate(items):
        image = Image.open(item["path"]).convert("RGB").resize((thumbnail, thumbnail), Image.Resampling.LANCZOS)
        grid.paste(image, (column * thumbnail, label_height))
        draw.text((column * thumbnail + 12, 16), f"Step {item['step']}", fill="black")
    grid_path = output_directory / f"case_{case_index + 1:02d}_checkpoint_grid.png"
    grid.save(grid_path)
    grid_paths.append(str(grid_path))
    master_rows.append(grid)
master_width = max(row.width for row in master_rows)
master_height = sum(row.height for row in master_rows)
master = Image.new("RGB", (master_width, master_height), "white")
y = 0
for row in master_rows:
    master.paste(row, (0, y))
    y += row.height
master_path = output_directory / "checkpoint_master_grid.png"
master.save(master_path)
metadata = {
    "checkpoint_steps": [int(record["step"]) for record in request["checkpoints"]],
    "checkpoint_selection_mode": request["mode"],
    "prompt_count": len(config["evaluation_prompts"]),
    "image_count": len(image_records),
    "character_scale": config["primary_adapter_scale"],
    "num_inference_steps": config["inference_steps"],
    "guidance_scale": config["inference_guidance"],
    "width": config["inference_width"],
    "height": config["inference_height"],
    "records": image_records,
    "per_prompt_grids": grid_paths,
    "master_grid": str(master_path),
    "automatic_best_checkpoint_selection": False,
    "permanent_merge_performed": False,
}
metadata_path = output_directory / "checkpoint_sweep_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("__KREA2_RESULT__" + json.dumps(metadata))
unload_runtime(model, transformer, controller, records)
"""
    environment = os.environ.copy()
    environment["KREA2_PROJECT_ROOT"] = str(PROJECT_ROOT)
    environment["KREA2_OUTPUT_DIRECTORY"] = str(output_directory)
    environment["PYTHONPATH"] = str(PROJECT_ROOT)
    environment["PYTHONUNBUFFERED"] = "1"
    result = subprocess.run(
        [str(PATHS["venv_python"]), "-c", script],
        cwd=str(PATHS["ai_toolkit"]),
        env=environment,
        capture_output=True,
        text=True,
    )
    log_path = PATHS["logs"] / "checkpoint_sweep.log"
    log_path.write_text(result.stdout + result.stderr + f"\nExit code: {result.returncode}\n", encoding="utf-8")
    if result.returncode != 0:
        raise RuntimeError(f"Checkpoint sweep failed.\n{result.stdout}\n{result.stderr}\nComplete log: {log_path}")
    structured = [line for line in result.stdout.splitlines() if line.startswith("__KREA2_RESULT__")]
    if len(structured) != 1:
        raise RuntimeError("The checkpoint sweep returned an invalid structured result.")
    metadata = json.loads(structured[0].removeprefix("__KREA2_RESULT__"))
    print(json.dumps({key: value for key, value in metadata.items() if key != "records"}, indent=2))
    display(Image.open(metadata["master_grid"]).convert("RGB"))

## Cell 23 — Apply an optional manual checkpoint selection

**Code cell.** After reviewing the dynamic sweep, set `manual_checkpoint_step` in the central configuration and run this cell. It updates only the active selection manifest. No checkpoint is treated as best without explicit user choice.

In [ ]:
import json
from pathlib import Path

manual_step = USER_CONFIG["manual_checkpoint_step"]
if manual_step is None:
    print("No manual checkpoint step is configured. The current active selection remains unchanged.")
else:
    inventory = json.loads((PATHS["config"] / "checkpoint_inventory.json").read_text(encoding="utf-8"))
    candidates = [record for record in inventory["checkpoints"] if record["step"] == manual_step]
    if len(candidates) != 1:
        raise RuntimeError(f"Manual checkpoint step {manual_step} was not found exactly once.")
    selected = candidates[0]
    selection = {
        "selection_mode": "manual_after_review",
        "source_kind": inventory["source_kind"],
        "training_complete": inventory["training_complete"],
        "final_quality_claim_allowed": inventory["final_quality_claim_allowed"],
        "checkpoint_step": selected["step"],
        "checkpoint_path": selected["path"],
        "checkpoint_sha256": selected["sha256"],
        "rank": selected["rank"],
        "adapter_name": USER_CONFIG["primary_adapter_name"],
        "adapter_scale": USER_CONFIG["primary_adapter_scale"],
        "permanent_merge_allowed": False,
    }
    selection_path = PATHS["config"] / "active_checkpoint_selection.json"
    selection_path.write_text(json.dumps(selection, indent=2), encoding="utf-8")
    print(json.dumps(selection, indent=2))

## Cell 24 — Run a configurable LoRA scale sweep

**Code cell.** Evaluates the active checkpoint at the editable list of scales while holding prompts, seeds, dimensions, inference steps, guidance, VAE, and optional additional LoRAs constant. It generates grids and metadata without modifying or merging the adapter.

In [ ]:
import json
import os
import subprocess
from pathlib import Path
from IPython.display import display
from PIL import Image

if not USER_CONFIG["run_inference"] or not USER_CONFIG["run_scale_sweep"]:
    print("LoRA scale sweep skipped by configuration.")
else:
    output_directory = PATHS["inference"] / "scale_sweep"
    output_directory.mkdir(parents=True, exist_ok=True)
    script = r"""
import json
import os
from pathlib import Path
from PIL import Image, ImageDraw
from toolkit.config_modules import GenerateImageConfig
from runtime_helpers.krea2_runtime import load_runtime, sha256_file, unload_runtime

root = Path(os.environ["KREA2_PROJECT_ROOT"])
config = json.loads((root / "config" / "user_configuration.json").read_text(encoding="utf-8"))
selection = json.loads((root / "config" / "active_checkpoint_selection.json").read_text(encoding="utf-8"))
assets = json.loads((root / "config" / "inference_asset_manifest.json").read_text(encoding="utf-8"))
output_directory = Path(os.environ["KREA2_OUTPUT_DIRECTORY"])
adapter_specs = [{
    "name": config["primary_adapter_name"],
    "path": selection["checkpoint_path"],
    "scale": config["primary_adapter_scale"],
    "alpha": config["lora_alpha"],
}] + config["additional_loras"]
model, transformer, controller, records = load_runtime(assets, adapter_specs, dtype=config["training_dtype"], max_text_length=config["max_text_length"])
image_records = []
paths_by_case = {index: [] for index in range(len(config["evaluation_prompts"]))}
for scale in config["scale_sweep"]:
    scales = {config["primary_adapter_name"]: float(scale)}
    for item in config["additional_loras"]:
        scales[item["name"]] = float(item.get("scale", 1.0))
    controller.set_scales(scales)
    configurations = []
    expected = []
    for case_index, prompt in enumerate(config["evaluation_prompts"]):
        seed = int(config["evaluation_seeds"][case_index % len(config["evaluation_seeds"])])
        scale_label = str(scale).replace(".", "p")
        output_path = output_directory / f"case_{case_index + 1:02d}_scale_{scale_label}.png"
        configurations.append(GenerateImageConfig(
            prompt=prompt,
            width=int(config["inference_width"]),
            height=int(config["inference_height"]),
            num_inference_steps=int(config["inference_steps"]),
            guidance_scale=float(config["inference_guidance"]),
            negative_prompt=config["negative_prompt"],
            seed=seed,
            network_multiplier=1.0,
            output_path=str(output_path),
            output_ext="png",
            add_prompt_file=False,
        ))
        expected.append((case_index, prompt, seed, output_path))
    model.generate_images(configurations)
    for case_index, prompt, seed, output_path in expected:
        if not output_path.is_file():
            raise RuntimeError(f"Expected scale-sweep image is missing: {output_path}")
        image_records.append({
            "case_index": case_index + 1,
            "scale": float(scale),
            "prompt": prompt,
            "seed": seed,
            "path": str(output_path),
            "sha256": sha256_file(output_path),
        })
        paths_by_case[case_index].append({"scale": float(scale), "path": str(output_path)})
thumbnail = 384
label_height = 48
grid_paths = []
master_rows = []
for case_index in range(len(config["evaluation_prompts"])):
    items = paths_by_case[case_index]
    grid = Image.new("RGB", (thumbnail * len(items), thumbnail + label_height), "white")
    draw = ImageDraw.Draw(grid)
    for column, item in enumerate(items):
        image = Image.open(item["path"]).convert("RGB").resize((thumbnail, thumbnail), Image.Resampling.LANCZOS)
        grid.paste(image, (column * thumbnail, label_height))
        draw.text((column * thumbnail + 12, 16), f"Scale {item['scale']}", fill="black")
    grid_path = output_directory / f"case_{case_index + 1:02d}_scale_grid.png"
    grid.save(grid_path)
    grid_paths.append(str(grid_path))
    master_rows.append(grid)
master_width = max(row.width for row in master_rows)
master_height = sum(row.height for row in master_rows)
master = Image.new("RGB", (master_width, master_height), "white")
y = 0
for row in master_rows:
    master.paste(row, (0, y))
    y += row.height
master_path = output_directory / "scale_master_grid.png"
master.save(master_path)
metadata = {
    "checkpoint_step": selection["checkpoint_step"],
    "checkpoint_path": selection["checkpoint_path"],
    "scales": [float(scale) for scale in config["scale_sweep"]],
    "prompt_count": len(config["evaluation_prompts"]),
    "image_count": len(image_records),
    "records": image_records,
    "per_prompt_grids": grid_paths,
    "master_grid": str(master_path),
    "permanent_merge_performed": False,
}
metadata_path = output_directory / "scale_sweep_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("__KREA2_RESULT__" + json.dumps(metadata))
unload_runtime(model, transformer, controller, records)
"""
    environment = os.environ.copy()
    environment["KREA2_PROJECT_ROOT"] = str(PROJECT_ROOT)
    environment["KREA2_OUTPUT_DIRECTORY"] = str(output_directory)
    environment["PYTHONPATH"] = str(PROJECT_ROOT)
    environment["PYTHONUNBUFFERED"] = "1"
    result = subprocess.run(
        [str(PATHS["venv_python"]), "-c", script],
        cwd=str(PATHS["ai_toolkit"]),
        env=environment,
        capture_output=True,
        text=True,
    )
    log_path = PATHS["logs"] / "scale_sweep.log"
    log_path.write_text(result.stdout + result.stderr + f"\nExit code: {result.returncode}\n", encoding="utf-8")
    if result.returncode != 0:
        raise RuntimeError(f"Scale sweep failed.\n{result.stdout}\n{result.stderr}\nComplete log: {log_path}")
    structured = [line for line in result.stdout.splitlines() if line.startswith("__KREA2_RESULT__")]
    if len(structured) != 1:
        raise RuntimeError("The scale sweep returned an invalid structured result.")
    metadata = json.loads(structured[0].removeprefix("__KREA2_RESULT__"))
    print(json.dumps({key: value for key, value in metadata.items() if key != "records"}, indent=2))
    display(Image.open(metadata["master_grid"]).convert("RGB"))

## Cell 25 — Package the current session dynamically

**Code cell.** Creates a core archive, dynamic checkpoint groups, a training-state archive when present, a selected-checkpoint copy, and a SHA-256 resume manifest. Reproducible large assets such as base models, the virtual environment, and the AI Toolkit checkout are intentionally excluded.

In [ ]:
import hashlib
import json
import math
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

export_directory = PATHS["exports"] / USER_CONFIG["run_name"]
if export_directory.exists():
    shutil.rmtree(export_directory)
export_directory.mkdir(parents=True, exist_ok=False)

inventory = json.loads((PATHS["config"] / "checkpoint_inventory.json").read_text(encoding="utf-8"))
selection = json.loads((PATHS["config"] / "active_checkpoint_selection.json").read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_zip(path, entries):
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for source, archive_name in entries:
            archive.write(source, arcname=str(archive_name))
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Archive creation failed: {path}")

excluded_roots = {PATHS["models"].resolve(), PATHS["venv"].resolve(), PATHS["ai_toolkit"].resolve(), PATHS["exports"].resolve(), PATHS["checkpoints"].resolve(), PATHS["smoke_checkpoints"].resolve()}
core_entries = []
for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file() or path.is_symlink():
        continue
    resolved = path.resolve()
    if any(str(resolved).startswith(str(root) + "/") or resolved == root for root in excluded_roots):
        continue
    core_entries.append((path, Path("project") / path.relative_to(PROJECT_ROOT)))

core_archive = export_directory / f"{USER_CONFIG['run_name']}_core.zip"
write_zip(core_archive, core_entries)

checkpoint_archives = []
checkpoint_records = inventory["checkpoints"]
chunk_size = USER_CONFIG["checkpoints_per_archive"]
for group_index in range(math.ceil(len(checkpoint_records) / chunk_size)):
    group = checkpoint_records[group_index * chunk_size:(group_index + 1) * chunk_size]
    archive_path = export_directory / f"{USER_CONFIG['run_name']}_checkpoints_{group_index + 1:02d}.zip"
    write_zip(archive_path, [(Path(record["path"]), Path("checkpoints") / Path(record["path"]).name) for record in group])
    checkpoint_archives.append(archive_path)

run_directory = Path(inventory["run_directory"])
training_state_files = [path for path in run_directory.rglob("*") if path.is_file() and path.suffix.lower() != ".safetensors"]
training_state_archive = None
if training_state_files:
    training_state_archive = export_directory / f"{USER_CONFIG['run_name']}_training_state.zip"
    write_zip(training_state_archive, [(path, Path("training_state") / path.relative_to(run_directory)) for path in training_state_files])

selected_source = Path(selection["checkpoint_path"])
selected_copy = export_directory / f"{USER_CONFIG['run_name']}_selected_step_{int(selection['checkpoint_step']):08d}.safetensors"
shutil.copy2(selected_source, selected_copy)
if sha256_file(selected_source) != sha256_file(selected_copy):
    raise RuntimeError("The selected checkpoint copy does not match its source.")

packages = [core_archive, *checkpoint_archives, selected_copy]
if training_state_archive is not None:
    packages.append(training_state_archive)
package_records = [
    {
        "filename": path.name,
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in packages
]
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_name": USER_CONFIG["project_name"],
    "run_name": USER_CONFIG["run_name"],
    "trigger_word": USER_CONFIG["trigger_word"],
    "source_kind": inventory["source_kind"],
    "training_complete": inventory["training_complete"],
    "final_quality_claim_allowed": inventory["final_quality_claim_allowed"],
    "checkpoint_steps": inventory["checkpoint_steps"],
    "selected_checkpoint": {
        "step": selection["checkpoint_step"],
        "source_path": selection["checkpoint_path"],
        "export_path": str(selected_copy),
        "sha256": sha256_file(selected_copy),
    },
    "packages": package_records,
    "excluded_reproducible_assets": [str(PATHS["models"]), str(PATHS["venv"]), str(PATHS["ai_toolkit"])],
}
manifest_path = export_directory / "resume_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
metadata_archive = export_directory / f"{USER_CONFIG['run_name']}_resume_metadata.zip"
write_zip(metadata_archive, [(manifest_path, Path("resume_manifest.json"))])
package_records.insert(0, {
    "filename": metadata_archive.name,
    "path": str(metadata_archive),
    "size_bytes": metadata_archive.stat().st_size,
    "sha256": sha256_file(metadata_archive),
})
manifest["packages"] = package_records
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"Export directory: {export_directory}")

## Cell 26 — Download export packages

**Code cell.** Downloads every package created by the previous cell only when `auto_download_exports` is enabled. Otherwise it prints the exact local paths so individual packages can be downloaded manually.

In [ ]:
import json
from pathlib import Path
from google.colab import files

export_directory = PATHS["exports"] / USER_CONFIG["run_name"]
manifest_path = export_directory / "resume_manifest.json"
if not manifest_path.is_file():
    raise RuntimeError("The export manifest is missing. Run the packaging cell first.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for package in manifest["packages"]:
    print(f"{package['filename']} | {package['size_bytes'] / (1024 ** 3):.3f} GiB | {package['sha256']}")
    if USER_CONFIG["auto_download_exports"]:
        files.download(package["path"])
if not USER_CONFIG["auto_download_exports"]:
    print("Automatic browser downloads are disabled. Set auto_download_exports to true or download the printed files manually.")

## Cell 27 — Optional generic restore utility

**Code cell.** Uploads one or more previously created generic export ZIP files and extracts their contents into a new temporary project root. It does not depend on a specific run name or predetermined archive numbering. Base models, AI Toolkit, and the virtual environment must still be recreated by the earlier cells.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import files

restore_root = Path("/content/krea2_lora_restore").resolve()
if restore_root.exists():
    shutil.rmtree(restore_root)
restore_root.mkdir(parents=True, exist_ok=False)

uploaded = files.upload()
zip_names = sorted(name for name in uploaded if name.lower().endswith(".zip"))
if not zip_names:
    raise RuntimeError("Upload at least one generic session ZIP archive.")

for name in zip_names:
    archive_path = restore_root / name
    archive_path.write_bytes(uploaded[name])
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            destination = (restore_root / "extracted" / member.filename).resolve()
            if not str(destination).startswith(str((restore_root / "extracted").resolve())):
                raise RuntimeError(f"Unsafe ZIP path detected: {member.filename}")
        archive.extractall(restore_root / "extracted")

manifests = list((restore_root / "extracted").rglob("resume_manifest.json"))
result = {
    "restore_root": str(restore_root),
    "uploaded_archives": zip_names,
    "resume_manifests": [str(path) for path in manifests],
    "base_models_restored": False,
    "environment_restored": False,
}
print(json.dumps(result, indent=2))

# Operating notes

- Edit the central configuration before running any later cell.
- Use an exact AI Toolkit commit for strict reproducibility. Using `main` is supported because the notebook records the resolved commit and package freeze.
- The installation cell deliberately uses `virtualenv`, not the Colab runtime's standard-library `venv` path.
- A broken or partially created `/content/krea2_lora/venv` directory is removed and rebuilt automatically.
- Dataset files must be matching image-caption pairs. Nested folders are accepted and canonicalized into a flat training directory.
- Trigger enforcement is configurable. Automatic caption modification is disabled by default and creates backups when enabled.
- Stop production training only after a checkpoint has finished saving. The checkpoint inventory cell validates whatever files actually exist.
- Smoke-only and interrupted checkpoints may be used for pipeline compatibility tests, but not final quality claims.
- A final-looking filename is not treated as independent proof that training completed.
- Checkpoint and scale sweeps use fixed prompts, seeds, dimensions, steps, guidance, VAE, and additional-LoRA settings for controlled comparisons.
- The notebook does not score identity or aesthetics automatically and does not select a best checkpoint.
- LoRAs remain separate adapters. Permanent merging is disabled.
- Live adapter scales are checked against their actual runtime-dtype representation, including BF16 rounding.
- Base model weights, the virtual environment, and the source checkout are excluded from export packages because their exact revisions are recorded for reconstruction.
